# Huấn luyện thêm: nhiều seed và biến thể, để thu hẹp khoảng cách với bài báo

Lần chạy 2 của đề tài đạt mIoU test 79,25 (1 seed), thấp hơn bài báo 3,65 điểm. Mô hình công bố của tác giả, chạy qua cùng code đánh giá, ra đúng 82,90, nên chênh lệch nằm ở khâu huấn luyện. Code huấn luyện của tác giả chỉ khác code đề tài ở lịch huấn luyện: 100 epoch cố định, không dừng sớm. Notebook này chạy các lần sau theo thứ tự ưu tiên, đều với batch 8 và ảnh 5 dải:

| Thứ tự | Tên | Lịch huấn luyện | Seed | Thời gian ước tính |
| --- | --- | --- | --- | --- |
| 1 | `e100_seed0` | 100 epoch cố định, learning rate không đổi | 0 | khoảng 29 phút |
| 2 | `e100_seed1` | 100 epoch cố định, learning rate không đổi | 1 | khoảng 29 phút |
| 3 | `e100_seed2` | 100 epoch cố định, learning rate không đổi | 2 | khoảng 29 phút |
| 4 | `e100_cosine_seed0` | 100 epoch cố định, learning rate giảm dần (cosine) | 0 | khoảng 29 phút |
| 5 | `es_seed1` | dừng sớm sau 30 epoch không cải thiện (tối đa 200 epoch), learning rate không đổi | 1 | tối đa khoảng 58 phút |
| 6 | `es_seed2` | dừng sớm sau 30 epoch không cải thiện (tối đa 200 epoch), learning rate không đổi | 2 | tối đa khoảng 58 phút |

Mỗi lần chạy: huấn luyện, rồi đánh giá trên tập test và chạy tầng quyết định cho **hai mô hình**: mô hình có mIoU val cao nhất (như lần chạy 2) và mô hình ở epoch cuối. Lý do: lần chạy 2 có val 83,94 nhưng test chỉ 79,25, còn mô hình của tác giả có val 79,53 và test 82,90; chọn epoch có val cao nhất trong nhiều epoch dao động có thể chọn trúng một epoch "may mắn" trên val. Mô hình epoch cuối của lần chạy 2 cũng được đánh giá ở đầu phiên đầu tiên (vài phút).

Tổng cộng tối đa khoảng 4–4,5 giờ GPU, tức **2 phiên Colab**.

**Cách chạy:** *Runtime → Change runtime type → T4 GPU*, rồi *Runtime → Run all*. Mỗi phiên tự dừng huấn luyện sau khoảng 3,5 giờ tính từ lúc máy Colab khởi động (chạy lại notebook trên cùng máy không làm đồng hồ đếm lại). Khi thấy dòng "Hết thời gian của phiên này", hôm sau mở lại notebook và **Run all lại**: lần đã xong tự bỏ qua, lần đang dở chạy tiếp từ checkpoint. Cuối notebook có lệnh ghi hết dữ liệu lên Drive; chờ nó xong rồi mới đóng phiên.

- Mỗi lần chạy chiếm khoảng 0,25 GB trên Drive sau khi xong (mô hình tốt nhất và xác suất dự đoán của hai mô hình); trong lúc chạy cần thêm khoảng 0,5 GB cho checkpoint, được xóa khi lần chạy xong. File xóa trên Drive nằm trong Thùng rác 30 ngày: nếu Drive gần đầy, hãy dọn Thùng rác.
- Giữ tab Colab mở trong lúc chạy. Nếu một bước bị lỗi, gửi thông báo lỗi cho Claude.

Kết quả nằm trong `MyDrive/De_Tai_Nong_San/weeds_pilot/runs/`. Mục cuối tạo file `ket_qua_weeds_runs.zip` để tải về.

## 1. Kết nối Drive và ghi mã nguồn (chạy ở **mọi** phiên)

In [ ]:
import os, sys, json
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')
BASE = Path('/content/drive/MyDrive/De_Tai_Nong_San/weeds_pilot')
DATA_DRIVE = BASE / 'data'
RUN = BASE / 'runs'  # one folder per run of src/weeds/runs.py; run 2 stays in run_dlv3p_msi_seed0_b8
RESULTS = Path('results/weeds_runs')  # folder inside the downloaded zip
for p in (DATA_DRIVE, RUN):
    p.mkdir(parents=True, exist_ok=True)

WORK = Path('/content/weeds_pilot')
FILES = {"src/__init__.py": "", "src/weeds/__init__.py": "\"\"\"Pilot on WeedsGalore (Celikkan et al., WACV 2025): crop/weed segmentation and a spraying decision layer.\"\"\"\n", "src/weeds/data.py": "\"\"\"Read the WeedsGalore dataset zip and convert each split into compact arrays.\n\nBand order R, G, B, NIR, RE and the 3-class merge (0 background, 1 maize, 2 any weed) follow the authors' loader\n(github.com/GFZ/weedsgalore, src/datasets/weedsgalore/weedsgalore.py). Images stay uint16; dividing by 65535 gives the\n[0, 1] scaling that their matplotlib-based reader produces.\n\"\"\"\nimport io\nimport zipfile\n\nimport numpy as np\nfrom PIL import Image\n\nROOT = 'weedsgalore-dataset'\nBANDS = ('R', 'G', 'B', 'NIR', 'RE')\nSPLITS = ('train', 'val', 'test')\nDATES = ('2023-05-25', '2023-05-30', '2023-06-06', '2023-06-15')\nPRE_HERBICIDE_DATES = DATES[:3]  # herbicide was applied on 12 June 2023 (Celikkan et al. 2025, 3.1)\nEXPECTED_TILES = {'train': 104, 'val': 26, 'test': 26}\nEXPECTED_TEST_LOCATIONS = 8  # 2 field patches x 4 sampled points (Celikkan et al. 2025, 3.2-3.3)\nTILE = 600\nSEMANTIC_CLASSES = ('background', 'maize', 'amaranth', 'barnyard_grass', 'quickweed', 'weed_other')\nCLASSES3 = ('background', 'crop', 'weed')\n# Celikkan et al. 2025, Table 3: DeepLabv3+ with 5-band (MSI) input, 3-class setting, test IoU in %\nPAPER_TEST_3CLASS_MSI = {'miou': 82.90, 'background': 98.45, 'crop': 72.93, 'weed': 77.31}\n\n\ndef split_ids(zf, split):\n    return zf.read(f'{ROOT}/splits/{split}.txt').decode().split()\n\n\ndef tile_date(tile_id):\n    return tile_id[:10]\n\n\ndef read_png(zf, name):\n    return np.array(Image.open(io.BytesIO(zf.read(name))))\n\n\ndef read_tile(zf, tile_id):\n    \"\"\"(5, 600, 600) uint16 image in BANDS order and (600, 600) uint8 semantic mask with values 0-5.\"\"\"\n    base = f'{ROOT}/{tile_date(tile_id)}'\n    bands = [read_png(zf, f'{base}/images/{tile_id}_{b}.png') for b in BANDS]\n    for b, a in zip(BANDS, bands):\n        if a.dtype != np.uint16:  # checked before stacking: an 8-bit band would otherwise be scaled by 65535 silently\n            raise ValueError(f'{tile_id} band {b}: expected 16-bit PNG, got {a.dtype}')\n    mask = read_png(zf, f'{base}/semantics/{tile_id}.png')\n    return np.stack(bands), mask.astype(np.uint8)\n\n\ndef location_blocks(ids):\n    \"\"\"Group the tile ids of a split file into field locations. In the val and test files each location is written\n    as a block of consecutive ids with decreasing dates (checked visually for the test split on 2026-10-29: shared\n    field markers and tyre tracks across dates). Returns a list of blocks.\"\"\"\n    blocks, cur = [], []\n    for t in ids:\n        if cur and tile_date(t) >= tile_date(cur[-1]):\n            blocks.append(cur)\n            cur = []\n        cur.append(t)\n    if cur:\n        blocks.append(cur)\n    return blocks\n\n\ndef to_3class(mask):\n    \"\"\"0 background, 1 maize, 2 weed (all weed species merged), as in the authors' binary_label.\"\"\"\n    out = mask.copy()\n    out[out > 1] = 2\n    return out\n\n\ndef load_split(zip_path, split):\n    with zipfile.ZipFile(zip_path) as zf:\n        ids = split_ids(zf, split)\n        tiles = [read_tile(zf, t) for t in ids]\n    x = np.stack([t[0] for t in tiles])\n    y = np.stack([t[1] for t in tiles])\n    check_split(split, ids, x, y)\n    return {'ids': np.array(ids), 'x': x, 'y': y}\n\n\ndef check_split(split, ids, x, y):\n    n = EXPECTED_TILES[split]\n    if len(ids) != n or x.shape != (n, len(BANDS), TILE, TILE) or y.shape != (n, TILE, TILE):\n        raise ValueError(f'{split}: unexpected shapes ids={len(ids)} x={x.shape} y={y.shape}')\n    if x.dtype != np.uint16 or y.max() > len(SEMANTIC_CLASSES) - 1:\n        raise ValueError(f'{split}: unexpected dtype {x.dtype} or label values up to {y.max()}')\n    if not set(tile_date(t) for t in ids) <= set(DATES):\n        raise ValueError(f'{split}: unknown acquisition date in ids')\n\n\ndef build_npz(zip_path, out_dir):\n    \"\"\"Write weedsgalore_<split>.npz (ids, x uint16, y uint8 semantic) for each split; returns the paths.\"\"\"\n    from pathlib import Path\n    out_dir = Path(out_dir)\n    out_dir.mkdir(parents=True, exist_ok=True)\n    paths = {}\n    for split in SPLITS:\n        d = load_split(zip_path, split)\n        paths[split] = out_dir / f'weedsgalore_{split}.npz'\n        np.savez_compressed(paths[split], **d)\n        print(f'{split}: {len(d[\"ids\"])} tiles, x {d[\"x\"].shape} {d[\"x\"].dtype}, saved {paths[split]}')\n    return paths\n\n\ndef load_npz(path):\n    with np.load(path) as d:\n        return {'ids': d['ids'].astype(str), 'x': d['x'], 'y': d['y']}\n", "src/weeds/model.py": "\"\"\"DeepLabv3+ with a ResNet-50 backbone, output stride 8, as used by the WeedsGalore authors\n(their code: VainF/DeepLabV3Plus-Pytorch). The ImageNet backbone comes from torchvision; the first convolution is\nreplaced by a randomly initialised 5-channel convolution for multispectral input, as the authors did.\nNeeds PyTorch (Colab).\"\"\"\nimport torch\nimport torch.nn.functional as F\nfrom torch import nn\nfrom torchvision.models import ResNet50_Weights, resnet50\n\n\nclass ASPPConv(nn.Sequential):\n    def __init__(self, cin, cout, dilation):\n        super().__init__(nn.Conv2d(cin, cout, 3, padding=dilation, dilation=dilation, bias=False),\n                         nn.BatchNorm2d(cout), nn.ReLU(inplace=True))\n\n\nclass ASPPPooling(nn.Sequential):\n    def __init__(self, cin, cout):\n        super().__init__(nn.AdaptiveAvgPool2d(1), nn.Conv2d(cin, cout, 1, bias=False),\n                         nn.BatchNorm2d(cout), nn.ReLU(inplace=True))\n\n    def forward(self, x):\n        size = x.shape[-2:]\n        return F.interpolate(super().forward(x), size=size, mode='bilinear', align_corners=False)\n\n\nclass ASPP(nn.Module):\n    def __init__(self, cin, rates, cout=256):\n        super().__init__()\n        branches = [nn.Sequential(nn.Conv2d(cin, cout, 1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True))]\n        branches += [ASPPConv(cin, cout, r) for r in rates]\n        branches.append(ASPPPooling(cin, cout))\n        self.branches = nn.ModuleList(branches)\n        self.project = nn.Sequential(nn.Conv2d(cout * len(branches), cout, 1, bias=False), nn.BatchNorm2d(cout),\n                                     nn.ReLU(inplace=True), nn.Dropout(0.1))\n\n    def forward(self, x):\n        return self.project(torch.cat([b(x) for b in self.branches], dim=1))\n\n\nclass DeepLabV3Plus(nn.Module):\n    def __init__(self, in_channels=5, num_classes=3, pretrained_backbone=True, rates=(12, 24, 36)):\n        super().__init__()\n        weights = ResNet50_Weights.IMAGENET1K_V1 if pretrained_backbone else None\n        r = resnet50(weights=weights, replace_stride_with_dilation=[False, True, True])  # output stride 8\n        if in_channels != 3:\n            r.conv1 = nn.Conv2d(in_channels, 64, kernel_size=7, stride=2, padding=3, bias=False)\n        self.stem = nn.Sequential(r.conv1, r.bn1, r.relu, r.maxpool)\n        self.layer1, self.layer2, self.layer3, self.layer4 = r.layer1, r.layer2, r.layer3, r.layer4\n        self.low_proj = nn.Sequential(nn.Conv2d(256, 48, 1, bias=False), nn.BatchNorm2d(48), nn.ReLU(inplace=True))\n        self.aspp = ASPP(2048, rates)\n        self.classifier = nn.Sequential(nn.Conv2d(304, 256, 3, padding=1, bias=False), nn.BatchNorm2d(256),\n                                        nn.ReLU(inplace=True), nn.Conv2d(256, num_classes, 1))\n        for m in [*self.low_proj.modules(), *self.aspp.modules(), *self.classifier.modules()]:\n            if isinstance(m, nn.Conv2d):\n                nn.init.kaiming_normal_(m.weight)\n            elif isinstance(m, nn.BatchNorm2d):\n                nn.init.ones_(m.weight)\n                nn.init.zeros_(m.bias)\n\n    def features(self, x):\n        \"\"\"Decoder features that feed the classifier (stride 4); Grad-CAM (src.weeds.explain) works on them.\"\"\"\n        low = self.layer1(self.stem(x))\n        high = self.layer4(self.layer3(self.layer2(low)))\n        out = F.interpolate(self.aspp(high), size=low.shape[-2:], mode='bilinear', align_corners=False)\n        return torch.cat([self.low_proj(low), out], dim=1)\n\n    def head(self, feat, size):\n        return F.interpolate(self.classifier(feat), size=size, mode='bilinear', align_corners=False)\n\n    def forward(self, x):\n        return self.head(self.features(x), x.shape[-2:])\n", "src/weeds/train.py": "\"\"\"Train DeepLabv3+ on WeedsGalore (3 classes, 5-band input), select it on validation mIoU, evaluate on test and save\nper-pixel class probabilities for the decision layer. Settings follow Celikkan et al. 2025 (4.1.1) and their code:\nAdam lr 1e-3, unweighted cross-entropy, rot90 / flip / Gaussian jitter (sd 1/25) augmentation, inputs scaled to [0, 1].\nThe learning rate is constant as in their code; --lr-schedule cosine is an optional variant.\nMixed precision is used to fit 600x600 tiles on a T4. Run on Colab GPU:\n\n  python -m src.weeds.train --data <dir with weedsgalore_<split>.npz> --out <dir> [--stage train|test|all] [--smoke]\n\nTraining resumes from <out>/last.pt and stops safely before --budget-hours.\n\"\"\"\nimport argparse\nimport json\nimport random\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\nimport torch.nn.functional as F\nfrom torch.utils.data import DataLoader, Dataset\n\nfrom src.weeds.data import CLASSES3, PAPER_TEST_3CLASS_MSI, SEMANTIC_CLASSES, load_npz, tile_date, to_3class\nfrom src.weeds.model import DeepLabV3Plus\n\n\nclass TileDataset(Dataset):\n    def __init__(self, d, augment):\n        self.x, self.y, self.augment = d['x'], to_3class(d['y']), augment\n\n    def __len__(self):\n        return len(self.x)\n\n    def __getitem__(self, i):\n        img = self.x[i].astype(np.float32) / 65535.0\n        lab = self.y[i]\n        if self.augment:\n            k = np.random.randint(4)\n            img, lab = np.rot90(img, k, axes=(1, 2)), np.rot90(lab, k)\n            flip = np.random.randint(3)  # 0 none, 1 vertical, 2 horizontal\n            if flip == 1:\n                img, lab = img[:, ::-1, :], lab[::-1, :]\n            elif flip == 2:\n                img, lab = img[:, :, ::-1], lab[:, ::-1]\n            img = img + np.random.randn(*img.shape).astype(np.float32) / 25\n        return (torch.from_numpy(np.ascontiguousarray(img)),\n                torch.from_numpy(np.ascontiguousarray(lab).astype(np.int64)))\n\n\ndef seed_worker(worker_id):\n    # each DataLoader worker would otherwise inherit the same numpy RNG state and repeat the same augmentations\n    s = torch.initial_seed() % 2 ** 32\n    np.random.seed(s)\n    random.seed(s)\n\n\ndef confusion(pred, lab, n=3):\n    \"\"\"cm[true, pred] accumulated over all pixels.\"\"\"\n    return torch.bincount((lab * n + pred).flatten(), minlength=n * n).reshape(n, n)\n\n\ndef iou_from_confusion(cm, names=CLASSES3):\n    \"\"\"Per-class IoU and mIoU in %. A class absent from both labels and predictions gets NaN and is left out of the\n    mIoU (torchmetrics, used by the authors, would count it as 0; every class occurs in the test split).\"\"\"\n    cm = cm.double()\n    tp = cm.diag()\n    denom = cm.sum(0) + cm.sum(1) - tp\n    iou = torch.where(denom > 0, tp / denom.clamp(min=1), torch.full_like(tp, float('nan')))\n    res = {c: float(100 * v) for c, v in zip(names, iou)}\n    res['miou'] = float(100 * torch.nanmean(iou))\n    return res\n\n\n@torch.no_grad()\ndef evaluate(model, d, dev, batch=2, keep_probs=False, limit=None, classes=3, bands=None, amp=True):\n    \"\"\"classes: 3 (labels merged by to_3class) or 6 (semantic labels); bands: use only the first `bands` input bands\n    (3 = R, G, B); amp: float16 autocast on GPU as in training, False = float32 as in the authors' evaluate.py.\n    keep_probs (3 classes only) also returns P(crop), P(weed) per pixel in float16.\"\"\"\n    if keep_probs and classes != 3:\n        raise ValueError('probabilities are kept for the 3-class setting only')\n    model.eval()\n    n = len(d['x']) if limit is None else min(limit, len(d['x']))\n    y = to_3class(d['y'][:n]) if classes == 3 else d['y'][:n]\n    cm = torch.zeros(classes, classes, dtype=torch.long, device=dev)\n    probs = []\n    for i in range(0, n, batch):\n        x = torch.from_numpy(d['x'][i:i + batch, :bands].astype(np.float32) / 65535.0).to(dev)\n        lab = torch.from_numpy(y[i:i + batch].astype(np.int64)).to(dev)\n        with torch.autocast('cuda', dtype=torch.float16, enabled=amp and dev.type == 'cuda'):\n            logits = model(x)\n        p = torch.softmax(logits.float(), dim=1)\n        cm += confusion(p.argmax(1), lab, classes)\n        if keep_probs:\n            probs.append(p[:, 1:].half().cpu().numpy())  # P(crop), P(weed)\n    res = iou_from_confusion(cm.cpu(), CLASSES3 if classes == 3 else SEMANTIC_CLASSES)\n    res['confusion'] = cm.cpu().tolist()\n    return (res, np.concatenate(probs)) if keep_probs else res\n\n\ndef torch_load(path, dev):\n    try:\n        return torch.load(path, map_location=dev, weights_only=False)  # our own checkpoint, not third-party\n    except TypeError:\n        return torch.load(path, map_location=dev)\n\n\ndef grad_scaler():\n    try:\n        return torch.amp.GradScaler('cuda')\n    except (AttributeError, TypeError):\n        return torch.cuda.amp.GradScaler()\n\n\ndef train(args, dev):\n    out = Path(args.out)\n    out.mkdir(parents=True, exist_ok=True)\n    if (out / 'train_done.json').exists():\n        print('training already finished:', (out / 'train_done.json').read_text())\n        return\n    tr, va = load_npz(Path(args.data) / 'weedsgalore_train.npz'), load_npz(Path(args.data) / 'weedsgalore_val.npz')\n    model = DeepLabV3Plus(5, 3, pretrained_backbone=not args.no_pretrained).to(dev)\n    opt = torch.optim.Adam(model.parameters(), lr=args.lr)\n    # 'constant' is the authors' setting; 'cosine' decays the learning rate to zero over args.epochs, one step per epoch\n    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=args.epochs) if args.lr_schedule == 'cosine' else None\n    scaler = grad_scaler()\n    # settings a resumed run must share with the checkpoint it resumes from\n    config = {k: getattr(args, k) for k in ('batch', 'lr', 'seed', 'epochs', 'patience', 'lr_schedule')}\n    gen = torch.Generator().manual_seed(args.seed)  # draws the shuffling order and the worker seeds; saved in last.pt\n    start, best, bad, history = 0, -1.0, 0, []\n    if (out / 'last.pt').exists():\n        ck = torch_load(out / 'last.pt', dev)\n        saved = ck.get('config')  # absent in checkpoints written before it was stored: constant schedule only\n        if (saved is None and args.lr_schedule != 'constant') or (saved is not None and saved != config):\n            raise ValueError(f'{out / \"last.pt\"} was written with settings {saved}, this run has {config}: '\n                             'use a new output folder')\n        model.load_state_dict(ck['model'])\n        opt.load_state_dict(ck['opt'])\n        scaler.load_state_dict(ck['scaler'])\n        if sched is not None:\n            sched.load_state_dict(ck['sched'])\n        if 'gen' in ck:\n            gen.set_state(ck['gen'].cpu())  # map_location moved it to the GPU; a CPU generator needs a CPU tensor\n        else:\n            gen.manual_seed(args.seed + ck['epoch'] + 1)  # how checkpoints without a saved generator state resumed\n        start, best, bad, history = ck['epoch'] + 1, ck['best_miou'], ck['bad'], ck['history']\n        print(f'resumed after epoch {ck[\"epoch\"]} (best val mIoU {best:.2f})')\n    if (out / 'best.json').exists():\n        saved_best = json.loads((out / 'best.json').read_text())['val_miou']\n        if saved_best > best:\n            # best.pt was written after last.pt, on epochs that resuming repeats with other random draws: keep that\n            # model (a worse one never replaces it) and count patience afresh from the resumed epoch\n            best, bad = saved_best, 0\n    loader = DataLoader(TileDataset(tr, augment=True), batch_size=args.batch, shuffle=True, drop_last=True,\n                        num_workers=args.workers, worker_init_fn=seed_worker, pin_memory=dev.type == 'cuda',\n                        generator=gen)\n    def save_last(epoch):\n        tmp = out / 'last.tmp'\n        torch.save({'model': model.state_dict(), 'opt': opt.state_dict(), 'scaler': scaler.state_dict(),\n                    'sched': sched.state_dict() if sched is not None else None, 'gen': gen.get_state(),\n                    'config': config, 'epoch': epoch, 'best_miou': best, 'bad': bad, 'history': history}, tmp)\n        tmp.replace(out / 'last.pt')\n\n    t0, finished, reason, epoch = time.time(), False, '', None\n    for epoch in range(start, args.epochs):\n        model.train()\n        te, losses, lr = time.time(), [], opt.param_groups[0]['lr']\n        for it, (x, y) in enumerate(loader):\n            x, y = x.to(dev, non_blocking=True), y.to(dev, non_blocking=True)\n            opt.zero_grad(set_to_none=True)\n            with torch.autocast('cuda', dtype=torch.float16, enabled=dev.type == 'cuda'):\n                loss = F.cross_entropy(model(x), y)\n            scaler.scale(loss).backward()\n            scaler.step(opt)\n            scaler.update()\n            losses.append(loss.item())\n            if args.max_iters and it + 1 >= args.max_iters:\n                break\n        if sched is not None:\n            sched.step()\n        val = evaluate(model, va, dev, limit=args.limit_val)\n        secs = time.time() - te\n        history.append({'epoch': epoch, 'lr': lr, 'train_loss': float(np.mean(losses)), 'val_miou': val['miou'],\n                        **{f'val_{c}': val[c] for c in CLASSES3}, 'seconds': secs})\n        if val['miou'] > best:\n            best, bad = val['miou'], 0\n            tmp = out / 'best.tmp'\n            torch.save(model.state_dict(), tmp)\n            tmp.replace(out / 'best.pt')\n            (out / 'best.json').write_text(json.dumps({'epoch': epoch, 'val_miou': best}))\n        else:\n            bad += 1\n        if (epoch + 1) % args.ckpt_every == 0:\n            save_last(epoch)\n        (out / 'history.json').write_text(json.dumps(history, indent=1))\n        mem = f' | peak GPU {torch.cuda.max_memory_allocated() / 1e9:.1f} GB' if dev.type == 'cuda' else ''\n        print(f'epoch {epoch:3d} | loss {history[-1][\"train_loss\"]:.4f} | val mIoU {val[\"miou\"]:.2f} '\n              f'(crop {val[\"crop\"]:.1f}, weed {val[\"weed\"]:.1f}) | best {best:.2f} | {secs:.0f} s{mem}', flush=True)\n        if bad >= args.patience:\n            finished, reason = True, f'early stopping: no val improvement for {args.patience} epochs'\n            break\n        # (the last epoch is done whatever the clock says: it ends the loop through the else below)\n        if epoch + 1 < args.epochs and time.time() - t0 + 1.5 * secs > args.budget_hours * 3600:\n            reason = f'time budget of {args.budget_hours} h reached: rerun to resume'\n            break\n    else:\n        finished, reason = True, f'reached {args.epochs} epochs'\n    if epoch is not None:\n        save_last(epoch)  # always checkpoint on exit, whatever stopped the loop\n    print(reason)\n    if finished and not args.smoke:\n        (out / 'train_done.json').write_text(json.dumps({'reason': reason, 'best_val_miou': best,\n                                                         'epochs_run': len(history), 'batch': args.batch,\n                                                         'lr': args.lr, 'seed': args.seed, 'epochs': args.epochs,\n                                                         'patience': args.patience,\n                                                         'lr_schedule': args.lr_schedule}))\n\n\ndef by_date(d, n):\n    \"\"\"The first n tiles of a split, grouped by acquisition date.\"\"\"\n    dates = [tile_date(t) for t in d['ids'][:n]]\n    return {day: {k: d[k][[i for i, x in enumerate(dates) if x == day]] for k in ('ids', 'x', 'y')}\n            for day in sorted(set(dates))}\n\n\ndef test(args, dev):\n    \"\"\"Evaluate the model chosen on validation (best.pt), or with --weights last the final-epoch model kept in\n    last.pt, whose outputs go to <out>/last/. Saves the probabilities for the decision layer and the test mIoU per\n    acquisition date.\"\"\"\n    out = Path(args.out)\n    model = DeepLabV3Plus(5, 3, pretrained_backbone=False).to(dev)\n    if args.weights == 'last':\n        ck = torch_load(out / 'last.pt', dev)\n        model.load_state_dict(ck['model'])\n        dest, info = out / 'last', {'weights': 'last', 'epoch': ck['epoch']}\n        dest.mkdir(exist_ok=True)\n    else:\n        model.load_state_dict(torch_load(out / 'best.pt', dev))\n        dest, info = out, {'weights': 'best', **json.loads((out / 'best.json').read_text())}\n    report = {'model': info}\n    for split in ('val', 'test'):\n        d = load_npz(Path(args.data) / f'weedsgalore_{split}.npz')\n        res, probs = evaluate(model, d, dev, keep_probs=True, limit=args.limit_val)\n        n = len(probs)\n        np.savez_compressed(dest / f'probs_{split}.npz', ids=d['ids'][:n], p_crop=probs[:, 0], p_weed=probs[:, 1])\n        report[split] = res\n        print(f'{split}: mIoU {res[\"miou\"]:.2f} | background {res[\"background\"]:.2f}, crop {res[\"crop\"]:.2f}, '\n              f'weed {res[\"weed\"]:.2f}')\n        if split == 'test':\n            report['test_by_date'] = {day: evaluate(model, sub, dev) for day, sub in by_date(d, n).items()}\n    report['paper_test_3class_msi'] = PAPER_TEST_3CLASS_MSI\n    report['test_minus_paper_miou'] = report['test']['miou'] - PAPER_TEST_3CLASS_MSI['miou']\n    (dest / 'segmentation_metrics.json').write_text(json.dumps(report, indent=1))\n    print(f'test mIoU {report[\"test\"][\"miou\"]:.2f} vs paper {PAPER_TEST_3CLASS_MSI[\"miou\"]:.2f} '\n          f'(diff {report[\"test_minus_paper_miou\"]:+.2f})')\n\n\ndef parser():\n    ap = argparse.ArgumentParser()\n    ap.add_argument('--data', required=True)\n    ap.add_argument('--out', required=True)\n    ap.add_argument('--stage', choices=['train', 'test', 'all'], default='all')\n    ap.add_argument('--weights', choices=['best', 'last'], default='best',\n                    help='test stage: the model chosen on validation, or the final-epoch model in last.pt')\n    ap.add_argument('--epochs', type=int, default=200)\n    ap.add_argument('--patience', type=int, default=30)\n    ap.add_argument('--batch', type=int, default=2, help='authors\\' code default; the paper reports 8 on an A100')\n    ap.add_argument('--ckpt-every', type=int, default=5, help='write last.pt (about 0.5 GB) every N epochs and on exit')\n    ap.add_argument('--lr', type=float, default=1e-3)\n    ap.add_argument('--lr-schedule', choices=['constant', 'cosine'], default='constant',\n                    help='constant as in the authors\\' code; cosine decays to zero over --epochs')\n    ap.add_argument('--workers', type=int, default=2)\n    ap.add_argument('--seed', type=int, default=0)\n    ap.add_argument('--budget-hours', type=float, default=3.5)\n    ap.add_argument('--max-iters', type=int, default=0, help='stop each epoch after this many iterations (smoke test)')\n    ap.add_argument('--limit-val', type=int, default=None, help='evaluate only the first N tiles (smoke test)')\n    ap.add_argument('--smoke', action='store_true', help='1 epoch of 2 iterations, 2 evaluation tiles')\n    ap.add_argument('--no-pretrained', action='store_true', help='random backbone (local tests only; skips download)')\n    return ap\n\n\ndef main():\n    args = parser().parse_args()\n    if args.smoke:\n        args.epochs, args.max_iters, args.limit_val, args.patience = 1, 2, 2, 1\n    random.seed(args.seed)\n    np.random.seed(args.seed)\n    torch.manual_seed(args.seed)\n    torch.backends.cudnn.benchmark = True\n    dev = torch.device('cuda' if torch.cuda.is_available() else 'cpu')\n    print('device:', dev, torch.cuda.get_device_name(0) if dev.type == 'cuda' else '')\n    if args.stage in ('train', 'all'):\n        train(args, dev)\n    if args.stage in ('test', 'all'):\n        test(args, dev)\n\n\nif __name__ == '__main__':\n    main()\n", "src/weeds/decision.py": "\"\"\"Spraying decision layer on top of the segmentation output (numpy only, runs anywhere).\n\nEach 600x600 tile is split into square cells of `cell` pixels (200 px = 0.5 m at the 2.5 mm GSD of WeedsGalore).\n  truth  w     : labelled weed fraction of the cell; the cell needs spraying if w >= tau\n  score  s     : mean predicted P(weed) over the cell, i.e. the expected weed fraction\nDecisions per cell: NO_SPRAY, SPRAY or DEFER (scout the cell; scouting is assumed to reveal the true need).\n\nClass-conditional split conformal thresholds, calibrated on held-out cells that played no part in training or model\nselection (Vovk et al.; as used per class by Melki et al. 2025 for weeding), give for an exchangeable new cell\n  P(s < lam_lo | need)   <= alpha   -> NO_SPRAY only when s < lam_lo   (missed spraying is bounded)\n  P(s > lam_hi | no need) <= beta   -> SPRAY only when s > lam_hi      (unneeded spraying is bounded)\nand cells in between are deferred. Cells of one tile are spatially correlated, so exchangeability, and with it the\nguarantee, holds only approximately; the empirical rates are reported next to alpha and beta.\n\"\"\"\nimport math\n\nimport numpy as np\nimport pandas as pd\n\nNO_SPRAY, SPRAY, DEFER = 0, 1, 2\n_EPS = 1e-9  # guards floor/ceil against floating-point error in alpha * (n + 1)\n\n\ndef cell_view(a, cell):\n    n, h, w = a.shape\n    if h % cell or w % cell:\n        raise ValueError(f'tile size {h}x{w} is not a multiple of the cell size {cell}')\n    return a.reshape(n, h // cell, cell, w // cell, cell)\n\n\ndef cell_table(ids, y3, p_weed, cell):\n    \"\"\"One row per cell: tile id, acquisition date, true weed and crop fractions, and the score s.\"\"\"\n    ids = np.asarray(ids)\n    if len(ids) != len(y3) or y3.shape != p_weed.shape:\n        raise ValueError(f'misaligned inputs: ids {len(ids)}, labels {y3.shape}, probabilities {p_weed.shape}')\n    w = cell_view(y3 == 2, cell).mean(axis=(2, 4))\n    crop = cell_view(y3 == 1, cell).mean(axis=(2, 4))\n    # float64: a float32 mean turns an exact 1 % cover (400 / 40000) into 0.0099999998 and flips a decision at tau\n    s = cell_view(p_weed.astype(np.float64), cell).mean(axis=(2, 4))\n    tile = np.repeat(np.arange(len(ids)), w.shape[1] * w.shape[2])\n    return pd.DataFrame({'tile': ids[tile], 'date': [t[:10] for t in ids[tile]], 'cell_size': cell,\n                         'w': w.ravel(), 'crop': crop.ravel(), 's': s.ravel()})\n\n\ndef lower_threshold(pos_scores, alpha):\n    \"\"\"floor(alpha (n+1))-th smallest score of calibration cells that need spraying; -inf when there are too few,\n    so that 'need' is never ruled out.\"\"\"\n    s = np.sort(np.asarray(pos_scores, dtype=float))\n    k = math.floor(alpha * (len(s) + 1) + _EPS)\n    return float(s[k - 1]) if k >= 1 else -np.inf\n\n\ndef upper_threshold(neg_scores, beta):\n    \"\"\"ceil((1-beta)(n+1))-th smallest score of calibration cells that do not need spraying; +inf when there are\n    too few, so that 'no need' is never ruled out.\"\"\"\n    s = np.sort(np.asarray(neg_scores, dtype=float))\n    k = math.ceil((1 - beta) * (len(s) + 1) - _EPS)\n    return float(s[k - 1]) if k <= len(s) else np.inf\n\n\ndef decide_conformal(s, lam_lo, lam_hi):\n    s = np.asarray(s, dtype=float)\n    need_possible = s >= lam_lo\n    no_need_possible = s <= lam_hi\n    d = np.full(s.shape, DEFER)\n    d[need_possible & ~no_need_possible] = SPRAY\n    d[~need_possible & no_need_possible] = NO_SPRAY\n    return d  # both possible, or neither (lam_hi < s < lam_lo): DEFER\n\n\ndef decide_threshold(s, t):\n    return np.where(np.asarray(s, dtype=float) >= t, SPRAY, NO_SPRAY)\n\n\ndef decision_metrics(d, need, w, crop):\n    d, need, w, crop = map(np.asarray, (d, need, w, crop))\n    spray, defer, no_spray = d == SPRAY, d == DEFER, d == NO_SPRAY\n    effective = spray | (defer & need)  # deferred cells are scouted and sprayed only if needed\n\n    def share(mask, among=None):\n        if among is None:\n            return float(mask.mean()) if mask.size else math.nan\n        return float(mask[among].mean()) if among.any() else math.nan\n\n    return {\n        'n_cells': int(d.size),\n        'infestation': share(need),\n        'sprayed': share(spray),\n        'deferred': share(defer),\n        'effective_sprayed': share(effective),\n        'spray_reduction': 1 - share(effective),\n        'missed_need_rate': share(no_spray, need),       # P(NO_SPRAY | needs spraying)\n        'false_spray_rate': share(spray, ~need),         # P(SPRAY | does not need spraying)\n        'missed_weed_area': float(w[no_spray].sum() / w.sum()) if w.sum() > 0 else math.nan,\n        'crop_area_sprayed': float(crop[spray].sum() / crop.sum()) if crop.sum() > 0 else math.nan,\n        'spray_surplus': share(spray) - share(need),     # Melki et al. 2025, eq. 13 (before scouting)\n    }\n\n\ndef policy_decisions(cal, test, tau, alpha, beta):\n    \"\"\"Thresholds from `cal` cells, decisions of every policy on `test` cells (DataFrames from cell_table).\"\"\"\n    need_cal = cal['w'].to_numpy() >= tau\n    s_cal = cal['s'].to_numpy()\n    lam_lo = lower_threshold(s_cal[need_cal], alpha)\n    lam_hi = upper_threshold(s_cal[~need_cal], beta)\n    s = test['s'].to_numpy()\n    need = test['w'].to_numpy() >= tau\n    policies = {\n        'broadcast': np.full(s.shape, SPRAY),\n        'oracle': np.where(need, SPRAY, NO_SPRAY),\n        'plug_in': decide_threshold(s, tau),                 # spray if the predicted weed fraction reaches tau\n        'conformal_miss_only': decide_threshold(s, lam_lo),  # spray unless 'need' is ruled out (no deferral)\n        'conformal_3way': decide_conformal(s, lam_lo, lam_hi),\n    }\n    info = {'lam_lo': lam_lo, 'lam_hi': lam_hi, 'n_cal_need': int(need_cal.sum()),\n            'n_cal_no_need': int((~need_cal).sum())}\n    return policies, info\n\n\ndef evaluate_policies(cal, test, tau, alpha, beta):\n    \"\"\"Calibrate on `cal` cells and evaluate every policy on `test` cells.\"\"\"\n    policies, info = policy_decisions(cal, test, tau, alpha, beta)\n    need = test['w'].to_numpy() >= tau\n    w, crop = test['w'].to_numpy(), test['crop'].to_numpy()\n    return [{'policy': k, 'tau': tau, 'alpha': alpha, 'beta': beta, **info, **decision_metrics(d, need, w, crop)}\n            for k, d in policies.items()]\n\n\ndef crossfit_decisions(cells, fold, tau, alpha, beta):\n    \"\"\"Decide each fold with thresholds calibrated on the other folds, so calibration cells never come from the\n    evaluated locations. `fold` gives one fold label per row of `cells`. Returns None when there is only one fold,\n    else (policies, per_row, per_fold): policies maps each policy to decisions aligned with the rows of `cells`;\n    per_row has the lam_lo / lam_hi applied to each row; per_fold lists policy_decisions' info in sorted fold order.\"\"\"\n    fold = np.asarray(fold)\n    folds = np.unique(fold)\n    if len(folds) < 2:\n        return None  # nothing left to calibrate on (e.g. a date whose tiles all fall in one fold)\n    policies, per_fold = {}, []\n    per_row = {'lam_lo': np.empty(len(cells)), 'lam_hi': np.empty(len(cells))}\n    for k in folds:\n        ev = fold == k\n        pol, info = policy_decisions(cells[~ev], cells[ev], tau, alpha, beta)\n        for name, d in pol.items():\n            policies.setdefault(name, np.empty(len(cells), dtype=int))[ev] = d\n        per_row['lam_lo'][ev], per_row['lam_hi'][ev] = info['lam_lo'], info['lam_hi']\n        per_fold.append(info)\n    return policies, per_row, per_fold\n\n\ndef evaluate_crossfit(cells, fold, tau, alpha, beta):\n    \"\"\"Every policy decided by crossfit_decisions, then all cells scored together. lam_lo / lam_hi are tuples of the\n    thresholds applied to each fold, in sorted fold order.\"\"\"\n    res = crossfit_decisions(cells, fold, tau, alpha, beta)\n    if res is None:\n        return []\n    policies, _, per_fold = res\n    need = cells['w'].to_numpy() >= tau\n    w, crop = cells['w'].to_numpy(), cells['crop'].to_numpy()\n    info = {'lam_lo': tuple(i['lam_lo'] for i in per_fold), 'lam_hi': tuple(i['lam_hi'] for i in per_fold),\n            'n_cal_need': min(i['n_cal_need'] for i in per_fold),\n            'n_cal_no_need': min(i['n_cal_no_need'] for i in per_fold)}\n    return [{'policy': name, 'tau': tau, 'alpha': alpha, 'beta': beta, **info, **decision_metrics(d, need, w, crop)}\n            for name, d in policies.items()]\n", "src/weeds/analyze.py": "\"\"\"Pilot decision analysis and go / no-go check (numpy and pandas only; runs on Colab or locally).\n\n  python -m src.weeds.analyze --data <dir with weedsgalore_<split>.npz> --out <dir with probs_*.npz, segmentation_metrics.json>\n\nThe validation split already chose the model (best epoch, early stopping), so it is not independent of the model.\nPrimary calibration is therefore 2-fold on the test split by field location: thresholds from 4 locations decide the\nother 4, and vice versa. Calibration on the validation split is reported as a secondary variant.\nThe weed-cover thresholds tau are illustrative until replaced by sourced action thresholds.\n\"\"\"\nimport argparse\nimport json\nimport math\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\n\nfrom src.weeds.data import (DATES, EXPECTED_TEST_LOCATIONS, PAPER_TEST_3CLASS_MSI, PRE_HERBICIDE_DATES,\n                            load_npz, location_blocks, to_3class)\nfrom src.weeds.decision import cell_table, crossfit_decisions, evaluate_crossfit, evaluate_policies\n\nCELL_SIZES = (100, 200)                # 0.25 m and 0.5 m cells at 2.5 mm GSD\nTAUS = (0.001, 0.005, 0.01, 0.02)      # illustrative weed-cover fractions that make a cell need spraying\nALPHA = BETA = 0.1\nSCOPES = {'pre-herbicide pooled': PRE_HERBICIDE_DATES, **{d: (d,) for d in DATES}}  # name -> acquisition dates\nCAL_TEST_2FOLD = 'test 2-fold by location'\nCAL_VAL = 'val (also used for model selection)'\nPRIMARY = {'cell_size': 200, 'tau': 0.01, 'policy': 'conformal_3way', 'scope': 'pre-herbicide pooled',\n           'calibration': CAL_TEST_2FOLD}\n# Proposed pilot criteria (agreed before running; adjustable). C1: test mIoU at most 5 points below the paper's value\n# for the same input; segmentation_metrics.json of the authors' checkpoints carries it as 'paper_test', our own runs\n# (5-band input) are compared with Table 3's MSI value.\nCRITERIA = {'miou_below_paper_max': 5.0, 'spray_reduction_min': 0.30, 'deferred_max': 0.30}\n\n\ndef load_cells(data_dir, out_dir, split, cell):\n    d = load_npz(Path(data_dir) / f'weedsgalore_{split}.npz')\n    with np.load(Path(out_dir) / f'probs_{split}.npz') as p:\n        ids, p_weed = p['ids'].astype(str), p['p_weed']\n    n = len(ids)\n    if not np.array_equal(ids, d['ids'][:n]):\n        raise ValueError(f'{split}: probabilities and labels are not in the same tile order')\n    return cell_table(ids, to_3class(d['y'][:n]), p_weed, cell)\n\n\ndef test_location_folds(test_ids):\n    \"\"\"Fold 0 / 1 per test tile, alternating over the 8 field locations (4 locations per fold).\"\"\"\n    blocks = location_blocks(test_ids)\n    if len(blocks) != EXPECTED_TEST_LOCATIONS:\n        raise ValueError(f'expected {EXPECTED_TEST_LOCATIONS} test locations, found {len(blocks)}')\n    return {t: i % 2 for i, b in enumerate(blocks) for t in b}\n\n\ndef primary_decisions(test_cells, fold_of):\n    \"\"\"Per-cell decisions of the primary setting: the test cells (cell_table rows at PRIMARY['cell_size']) of\n    PRIMARY['scope'], each with its fold, the decision of PRIMARY['policy'], the thresholds applied to it from the\n    2-fold cross-fit by location, and its index within the tile ('cell', row-major). fold_of: test_location_folds.\"\"\"\n    if PRIMARY['calibration'] != CAL_TEST_2FOLD or set(test_cells['cell_size']) != {PRIMARY['cell_size']}:\n        raise ValueError('primary_decisions expects test cells of the primary cell size, cross-fit calibration')\n    t = test_cells[test_cells['date'].isin(SCOPES[PRIMARY['scope']])].copy()\n    t['fold'] = t['tile'].map(fold_of).to_numpy()\n    policies, per_row, _ = crossfit_decisions(t, t['fold'].to_numpy(), PRIMARY['tau'], ALPHA, BETA)\n    t['decision'] = policies[PRIMARY['policy']]\n    t['lam_lo'], t['lam_hi'] = per_row['lam_lo'], per_row['lam_hi']\n    t['cell'] = t.groupby('tile').cumcount()\n    return t.reset_index(drop=True)\n\n\ndef run(data_dir, out_dir):\n    rows = []\n    for cell in CELL_SIZES:\n        val_all, test_all = load_cells(data_dir, out_dir, 'val', cell), load_cells(data_dir, out_dir, 'test', cell)\n        fold_of = test_location_folds(list(dict.fromkeys(test_all['tile'])))\n        for tau in TAUS:\n            for scope, dates in SCOPES.items():\n                t = test_all[test_all['date'].isin(dates)]\n                if t.empty:\n                    continue\n                base = {'cell_size': cell, 'scope': scope}\n                rows += [{**base, 'calibration': CAL_TEST_2FOLD, **r}\n                         for r in evaluate_crossfit(t, t['tile'].map(fold_of).to_numpy(), tau, ALPHA, BETA)]\n                v = val_all[val_all['date'].isin(dates)]\n                if not v.empty:\n                    rows += [{**base, 'calibration': CAL_VAL, **r} for r in evaluate_policies(v, t, tau, ALPHA, BETA)]\n    return pd.DataFrame(rows)\n\n\ndef pick(df, **keys):\n    m = np.ones(len(df), dtype=bool)\n    for k, v in keys.items():\n        m &= (df[k] == v).to_numpy()\n    if m.sum() != 1:\n        raise ValueError(f'expected exactly one row for {keys}, found {m.sum()}')\n    return df[m].iloc[0]\n\n\ndef go_no_go(df, seg):\n    key = {k: PRIMARY[k] for k in ('cell_size', 'tau', 'scope', 'calibration')}\n    r = pick(df, policy=PRIMARY['policy'], **key)\n    oracle = pick(df, policy='oracle', **key)\n    miou = seg['test']['miou']\n    miou_min = seg.get('paper_test', PAPER_TEST_3CLASS_MSI)['miou'] - CRITERIA['miou_below_paper_max']\n    checks = {\n        'C1_test_miou': {'value': miou, 'required': f'>= {miou_min:.2f}', 'pass': bool(miou >= miou_min)},\n        'C2_spray_reduction': {'value': r['spray_reduction'], 'required': f\">= {CRITERIA['spray_reduction_min']}\",\n                               'pass': bool(r['spray_reduction'] >= CRITERIA['spray_reduction_min'])},\n        'C3_deferred': {'value': r['deferred'], 'required': f\"<= {CRITERIA['deferred_max']}\",\n                        'pass': bool(r['deferred'] <= CRITERIA['deferred_max'])},\n    }\n    info = {\n        'primary_setting': PRIMARY, 'alpha': ALPHA, 'beta': BETA,\n        'missed_need_rate': r['missed_need_rate'], 'false_spray_rate': r['false_spray_rate'],\n        'infestation': r['infestation'], 'oracle_spray_reduction': oracle['spray_reduction'],\n        'c2_reachable': bool(oracle['spray_reduction'] >= CRITERIA['spray_reduction_min']),\n        'n_cal_need': int(r['n_cal_need']), 'n_cal_no_need': int(r['n_cal_no_need']),\n        'lam_lo': r['lam_lo'], 'lam_hi': r['lam_hi'],\n    }\n    return {'verdict': 'GO' if all(c['pass'] for c in checks.values()) else 'NO-GO', 'checks': checks, 'info': info}\n\n\ndef clean(o):\n    \"\"\"JSON-safe: NaN / inf -> None, numpy scalars -> Python.\"\"\"\n    if isinstance(o, dict):\n        return {k: clean(v) for k, v in o.items()}\n    if isinstance(o, (list, tuple)):\n        return [clean(v) for v in o]\n    if isinstance(o, (np.floating, float)):\n        return None if not math.isfinite(float(o)) else float(o)\n    if isinstance(o, np.integer):\n        return int(o)\n    if isinstance(o, np.bool_):\n        return bool(o)\n    return o\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument('--data', required=True)\n    ap.add_argument('--out', required=True)\n    args = ap.parse_args()\n    out = Path(args.out)\n    df = run(args.data, out)\n    df.to_csv(out / 'decision_metrics.csv', index=False)\n    seg = json.loads((out / 'segmentation_metrics.json').read_text())\n    verdict = go_no_go(df, seg)\n    (out / 'go_no_go.json').write_text(json.dumps(clean(verdict), indent=1))\n\n    cols = ['policy', 'infestation', 'sprayed', 'deferred', 'spray_reduction', 'missed_need_rate', 'false_spray_rate',\n            'missed_weed_area', 'crop_area_sprayed']\n    same = (df['cell_size'] == PRIMARY['cell_size']) & (df['tau'] == PRIMARY['tau'])\n    pd.set_option('display.width', 200)\n    for cal in (CAL_TEST_2FOLD, CAL_VAL):\n        print(f\"\\nCell {PRIMARY['cell_size']} px, tau {PRIMARY['tau']}, {PRIMARY['scope']}, calibration: {cal}, \"\n              f'alpha = beta = {ALPHA}')\n        print(df[same & (df['scope'] == PRIMARY['scope']) & (df['calibration'] == cal)][cols].round(3)\n              .to_string(index=False))\n    per_date = df[same & (df['policy'] == PRIMARY['policy']) & (df['scope'] != PRIMARY['scope'])]\n    print('\\nPer acquisition date (conformal_3way):')\n    print(per_date[['scope', 'calibration', 'n_cal_need', 'infestation', 'deferred', 'spray_reduction',\n                    'missed_need_rate', 'false_spray_rate']].round(3).to_string(index=False))\n    print('\\nGO / NO-GO:', verdict['verdict'])\n    for name, c in verdict['checks'].items():\n        print(f\"  {name}: {c['value']:.3f} (required {c['required']}) -> {'PASS' if c['pass'] else 'FAIL'}\")\n    i = verdict['info']\n    print(f\"  validity: missed-need rate {i['missed_need_rate']:.3f} vs alpha {ALPHA}, \"\n          f\"false-spray rate {i['false_spray_rate']:.3f} vs beta {BETA}\")\n    if not i['c2_reachable']:\n        print(f\"  note: C2 cannot be met at this tau even by the oracle (oracle reduction {i['oracle_spray_reduction']:.3f})\")\n\n\nif __name__ == '__main__':\n    main()\n", "src/weeds/authors.py": "\"\"\"Evaluate the DeepLabv3+ checkpoints released by the WeedsGalore authors (ckpts.zip on the dataset DOI page) with\nthe evaluation code of this project (src.weeds.train.evaluate, the function that scored our own runs). Two uses:\ncheck our evaluation against the paper (Celikkan et al. 2025, Tables 3 and 4), and run the decision layer on the\nauthors' own model next to ours.\n\nTheir network is VainF's DeepLabV3Plus with the first convolution replaced to take `in_channels` bands (their\nsrc/nets/deeplabv3plus and src/evaluate.py); src.weeds.model has the same layers under other names, so the state dict\nis renamed, never converted, and loaded with strict=True. Inputs are scaled to [0, 1] and RGB uses the first three\nbands (R, G, B), as in their dataset class. Each checkpoint is evaluated twice:\n  float32  as in the authors' evaluate.py: the numbers to compare with the paper;\n  float16  autocast, as our runs were evaluated: shows what float16 inference alone changes, and gives the\n           probabilities for the decision layer, so it treats both models alike.\nRun on Colab GPU (on CPU both modes are float32):\n\n  python -m src.weeds.authors --data <dir with weedsgalore_<split>.npz> --ckpts <dir with dlv3p_*.pth> --out <dir>\n\nFor each checkpoint <out>/<name>/ gets segmentation_metrics.json ('val' and 'test' from float16 inference, 'fp32' with\nboth splits in float32); 3-class checkpoints also get probs_val.npz and probs_test.npz in the format of\nsrc.weeds.train, so `python -m src.weeds.analyze --out <out>/<name>` works on them.\n\"\"\"\nimport argparse\nimport json\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\n\nfrom src.weeds.analyze import clean\nfrom src.weeds.data import BANDS, PAPER_TEST_3CLASS_MSI, load_npz\nfrom src.weeds.model import DeepLabV3Plus\nfrom src.weeds.train import evaluate\n\nCKPT_URL = 'https://doidata.gfz.de/weedsgalore_e_celikkan_2024/ckpts.zip'\nCKPT_ZIP_BYTES = 886908486  # size served by GFZ on 2026-11-05\n# deterministic checkpoints in ckpts.zip: name -> (input bands, classes); the dlv3p_do_* (MC dropout) are not used\nCKPTS = {'dlv3p_msi_3': (5, 3), 'dlv3p_rgb_3': (3, 3), 'dlv3p_msi_6': (5, 6), 'dlv3p_rgb_6': (3, 6)}\n# Celikkan et al. 2025, DeepLabv3+, test IoU in %: Table 3 (3 classes) and Table 4 (6 classes)\nPAPER = {\n    'dlv3p_msi_3': PAPER_TEST_3CLASS_MSI,\n    'dlv3p_rgb_3': {'miou': 79.33, 'background': 97.97, 'crop': 67.93, 'weed': 72.08},\n    'dlv3p_msi_6': {'miou': 55.52, 'background': 98.37, 'maize': 73.03, 'amaranth': 76.17, 'barnyard_grass': 53.55,\n                    'quickweed': 21.11, 'weed_other': 10.86},\n    'dlv3p_rgb_6': {'miou': 50.81, 'background': 97.94, 'maize': 71.46, 'amaranth': 74.32, 'barnyard_grass': 45.95,\n                    'quickweed': 6.26, 'weed_other': 8.92},\n}\n# their parameter-name prefixes -> ours (src.weeds.model)\nRENAMES = [('backbone.conv1.', 'stem.0.'), ('backbone.bn1.', 'stem.1.'), ('backbone.layer', 'layer'),\n           ('classifier.project.', 'low_proj.'), ('classifier.aspp.convs.', 'aspp.branches.'),\n           ('classifier.aspp.project.', 'aspp.project.'), ('classifier.classifier.', 'classifier.')]\n\n\ndef rename_state_dict(sd):\n    out = {}\n    for k, v in sd.items():\n        name = k[len('module.'):] if k.startswith('module.') else k\n        for old, new in RENAMES:\n            if name.startswith(old):\n                out[new + name[len(old):]] = v\n                break\n        else:\n            raise KeyError(f'unexpected parameter name in checkpoint: {k}')\n    return out\n\n\ndef load_checkpoint(path, bands, classes, dev):\n    # weights_only: a third-party file must not be able to run code when unpickled\n    sd = torch.load(path, map_location='cpu', weights_only=True)\n    model = DeepLabV3Plus(bands, classes, pretrained_backbone=False)\n    model.load_state_dict(rename_state_dict(sd), strict=True)\n    return model.to(dev).eval()\n\n\ndef per_class(res):\n    return {k: v for k, v in res.items() if k != 'confusion'}\n\n\ndef run(data_dir, ckpt_dir, out_dir, dev):\n    if BANDS[:3] != ('R', 'G', 'B'):\n        raise ValueError('RGB checkpoints expect the first three bands to be R, G, B')\n    missing = [n for n in CKPTS if not (Path(ckpt_dir) / f'{n}.pth').exists()]\n    if missing:\n        raise FileNotFoundError(f'checkpoints not found in {ckpt_dir}: {missing}')\n    splits = {s: load_npz(Path(data_dir) / f'weedsgalore_{s}.npz') for s in ('val', 'test')}\n    summary = {}\n    for name, (bands, classes) in CKPTS.items():\n        model = load_checkpoint(Path(ckpt_dir) / f'{name}.pth', bands, classes, dev)\n        out = Path(out_dir) / name\n        out.mkdir(parents=True, exist_ok=True)\n        report = {'fp32': {}}\n        for split, d in splits.items():\n            report['fp32'][split] = evaluate(model, d, dev, classes=classes, bands=bands, amp=False)\n            if classes == 3:\n                res, probs = evaluate(model, d, dev, keep_probs=True, classes=3, bands=bands, amp=True)\n                np.savez_compressed(out / f'probs_{split}.npz', ids=d['ids'], p_crop=probs[:, 0], p_weed=probs[:, 1])\n            else:\n                res = evaluate(model, d, dev, classes=classes, bands=bands, amp=True)\n            report[split] = res\n        paper = PAPER[name]\n        report['paper_test'] = paper\n        report['fp32_test_minus_paper_miou'] = report['fp32']['test']['miou'] - paper['miou']\n        report['test_minus_paper_miou'] = report['test']['miou'] - paper['miou']\n        (out / 'segmentation_metrics.json').write_text(json.dumps(clean(report), indent=1))\n        summary[name] = {'bands': bands, 'classes': classes, 'paper': paper,\n                         'fp32_test': per_class(report['fp32']['test']), 'fp16_test': per_class(report['test']),\n                         'fp32_test_minus_paper_miou': report['fp32_test_minus_paper_miou'],\n                         'fp16_test_minus_paper_miou': report['test_minus_paper_miou']}\n        t32, t16 = report['fp32']['test'], report['test']\n        print(f'{name}: test mIoU float32 {t32[\"miou\"]:.2f}, float16 {t16[\"miou\"]:.2f}, paper {paper[\"miou\"]:.2f} '\n              f'(float32 - paper {report[\"fp32_test_minus_paper_miou\"]:+.2f}) | float32 / paper: ' +\n              ', '.join(f'{c} {t32[c]:.2f} / {v:.2f}' for c, v in paper.items() if c != 'miou'))\n        del model\n    Path(out_dir).mkdir(parents=True, exist_ok=True)\n    (Path(out_dir) / 'authors_vs_paper.json').write_text(json.dumps(clean(summary), indent=1))\n    return summary\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument('--data', required=True)\n    ap.add_argument('--ckpts', required=True)\n    ap.add_argument('--out', required=True)\n    args = ap.parse_args()\n    dev = torch.device('cuda' if torch.cuda.is_available() else 'cpu')\n    print('device:', dev, torch.cuda.get_device_name(0) if dev.type == 'cuda' else '')\n    run(args.data, args.ckpts, args.out, dev)\n\n\nif __name__ == '__main__':\n    main()\n", "src/weeds/runs.py": "\"\"\"Further training runs to explain the gap between our pilot run 2 (test mIoU 79.25, one seed) and the paper (82.90),\nwhose released checkpoint our evaluation reproduces exactly (src.weeds.authors). The authors' training code differs\nfrom ours only in its schedule: 100 epochs without early stopping, checkpoints chosen on validation. So the runs\nmeasure seed variance under both schedules, plus one variant with a decaying learning rate. All use batch 8 and the\n5-band input. Each run is evaluated twice: with the model chosen on validation (best.pt), and with the final-epoch\nmodel, because choosing the best of many noisy validation scores may pick a lucky epoch (run 2: validation 83.94,\ntest 79.25; the authors' released model: validation 79.53, test 82.90).\nnotebooks/colab_weeds_runs.ipynb trains them in the order below, as many as fit in a Colab session, evaluates each,\nruns the decision layer, and resumes the rest in the next session.\n\"\"\"\nimport json\nfrom pathlib import Path\n\nimport pandas as pd\n\nfrom src.weeds.analyze import PRIMARY, pick\n\nBATCH = 8\nRUNS = [  # in order of priority\n    # the authors' schedule (github.com/GFZ/weedsgalore README: --epochs 100), constant learning rate\n    {'name': 'e100_seed0', 'seed': 0, 'epochs': 100, 'patience': 100, 'lr_schedule': 'constant'},\n    {'name': 'e100_seed1', 'seed': 1, 'epochs': 100, 'patience': 100, 'lr_schedule': 'constant'},\n    {'name': 'e100_seed2', 'seed': 2, 'epochs': 100, 'patience': 100, 'lr_schedule': 'constant'},\n    # 100 epochs with the learning rate decayed to zero: a steadier end of training (not in the authors' code)\n    {'name': 'e100_cosine_seed0', 'seed': 0, 'epochs': 100, 'patience': 100, 'lr_schedule': 'cosine'},\n    # our pilot schedule (run 2 = seed 0): early stopping after 30 epochs without improvement, at most 200\n    {'name': 'es_seed1', 'seed': 1, 'epochs': 200, 'patience': 30, 'lr_schedule': 'constant'},\n    {'name': 'es_seed2', 'seed': 2, 'epochs': 200, 'patience': 30, 'lr_schedule': 'constant'},\n]\n# what run 2 used: its train_done.json predates these fields (src.weeds.train defaults on 2026-11-04)\nLEGACY = {'epochs': 200, 'patience': 30, 'lr_schedule': 'constant'}\n\n\ndef train_args(run):\n    \"\"\"Command-line arguments of src.weeds.train for one run (data, output and budget are added by the caller).\"\"\"\n    return ['--batch', str(BATCH), '--seed', str(run['seed']), '--epochs', str(run['epochs']),\n            '--patience', str(run['patience']), '--lr-schedule', run['lr_schedule']]\n\n\ndef fixed(run):\n    \"\"\"True when early stopping can never fire before the last epoch.\"\"\"\n    return run['patience'] >= run['epochs']\n\n\ndef schedule(done):\n    \"\"\"Schedule label from a train_done.json (or a RUNS entry).\"\"\"\n    s = {**LEGACY, **{k: done[k] for k in LEGACY if k in done}}\n    if s['lr_schedule'] != 'constant':\n        return f\"{s['epochs']} epochs, {s['lr_schedule']} lr\"\n    return f\"{s['epochs']} epochs fixed\" if fixed(s) else f\"early stopping ({s['patience']})\"\n\n\ndef decision_row(csv):\n    r = pick(pd.read_csv(csv), policy=PRIMARY['policy'],\n             **{k: PRIMARY[k] for k in ('cell_size', 'tau', 'scope', 'calibration')})\n    return {k: r[k] for k in ('spray_reduction', 'deferred', 'missed_need_rate', 'false_spray_rate')}\n\n\ndef summarize(run_dirs):\n    \"\"\"One row per evaluated model of each finished run: 'best val' (best.pt) and, when <run>/last/ was evaluated,\n    'last epoch'. Columns: schedule, batch, seed, epoch of the model, validation and test mIoU, and the decisions of\n    conformal_3way in the primary setting. Runs without results are listed in df.attrs['skipped'].\"\"\"\n    rows, skipped = [], []\n    for name, d in run_dirs.items():\n        d = Path(d)\n        need = [d / f for f in ('train_done.json', 'best.json', 'segmentation_metrics.json', 'decision_metrics.csv')]\n        if not all(f.exists() for f in need):\n            skipped.append(name)\n            continue\n        done, best = json.loads(need[0].read_text()), json.loads(need[1].read_text())\n        base = {'run': name, 'schedule': schedule(done), 'batch': done['batch'], 'seed': done['seed'],\n                'epochs_run': done['epochs_run']}\n        models = [('best val', best['epoch'], d)]\n        last = d / 'last'\n        if (last / 'segmentation_metrics.json').exists() and (last / 'decision_metrics.csv').exists():\n            models.append(('last epoch', json.loads((last / 'segmentation_metrics.json').read_text())['model']['epoch'],\n                           last))\n        for selection, epoch, m in models:\n            seg = json.loads((m / 'segmentation_metrics.json').read_text())\n            rows.append({**base, 'selection': selection, 'epoch': epoch, 'val_miou': seg['val']['miou'],\n                         'test_miou': seg['test']['miou'], 'test_crop': seg['test']['crop'],\n                         'test_weed': seg['test']['weed'], **decision_row(m / 'decision_metrics.csv')})\n    df = pd.DataFrame(rows)\n    df.attrs['skipped'] = skipped\n    return df\n\n\ndef by_schedule(df):\n    \"\"\"Count, mean, sample standard deviation (NaN for a single run) and range of test mIoU per schedule, batch\n    size and model selection.\"\"\"\n    keys = ['schedule', 'batch', 'selection']\n    if df.empty:\n        return pd.DataFrame(columns=keys + ['n', 'mean', 'std', 'min', 'max'])\n    return df.groupby(keys)['test_miou'].agg(n='count', mean='mean', std='std', min='min', max='max').reset_index()\n", "src/weeds/explain.py": "\"\"\"Explanations of the per-cell spraying decisions, and a test of whether they are faithful.\n\nA cell's decision depends only on its score s = mean P(weed) over the cell (src.weeds.decision), so explaining the\ndecision means explaining s: which input pixels and which spectral bands raise it. The model has a large receptive\nfield, so pixels outside the cell can matter too; attributions cover the whole tile.\n\nAttributions of s, compared:\n  integrated_gradients   Sundararajan et al. 2017 (ICML), from a 'bare soil' baseline: every band of every pixel set\n                         to the band's median over background pixels of the training split. Attributions per pixel\n                         and band sum to s(x) - s(baseline) (completeness, checked and reported).\n  grad_cam               Selvaraju et al. 2017 (ICCV), on the decoder features (stride 4) that feed the classifier.\n  predicted_weed         the model's own P(weed) inside the cell, a reference that needs no explanation method.\nBand importance: each band in turn is replaced by its bare-soil value over the whole tile, and the drop in s recorded.\nGradients are taken of the cell's SUM of P(weed) (cell^2 times s) and scaled back: under float16 autocast the\ngradient of a 40 000-pixel mean underflows to zero.\n\nFaithfulness (deletion test, Petsiuk et al. 2018, with random controls), for the SPRAY and DEFER cells: the patches\nan explanation ranks highest are replaced by bare soil and s is recomputed; the same number of patches drawn at\nrandom over the whole tile, and inside the cell, gives the controls. A faithful explanation removes the weed evidence:\ns drops more than under the random controls, and SPRAY decisions flip sooner. Run on Colab GPU:\n\n  python -m src.weeds.explain --data <dir with weedsgalore_<split>.npz> --ckpt <dlv3p_msi_3.pth> --out <dir>\n                              [--expect <go_no_go.json reported for this model>]\n\nThe decisions are computed once (decisions.csv) and reused by later sessions; per-tile results go to <out>/parts/, so\nan interrupted run resumes, and settings are fixed per output folder (xai_config.json). When every tile is done the\nparts are merged into xai_cells.csv, xai_deletion.csv, xai_maps.npz and xai_summary.json.\n\"\"\"\nimport argparse\nimport json\nimport math\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport torch\nimport torch.nn.functional as F\n\nfrom src.weeds.analyze import PRIMARY, clean, primary_decisions, test_location_folds\nfrom src.weeds.authors import load_checkpoint\nfrom src.weeds.data import BANDS, load_npz, to_3class\nfrom src.weeds.decision import DEFER, NO_SPRAY, SPRAY, cell_table, decide_conformal, decision_metrics\nfrom src.weeds.train import evaluate\n\nXAI_VERSION = 2  # bump when an explanation or the deletion test changes: parts of another version are not merged\nWEED = 2\nMETHODS = ('integrated_gradients', 'grad_cam', 'predicted_weed')\nCONTROLS = ('random_tile', 'random_cell')\nFRACTIONS = (0.005, 0.01, 0.02, 0.05, 0.1)  # share of the tile's patches removed in the deletion test\nMAP_POOL = 4  # attribution maps are stored average-pooled 4 x 4 (600 -> 150 px)\n\n\ndef autocast(x, amp):\n    return torch.autocast('cuda', dtype=torch.float16, enabled=amp and x.is_cuda)\n\n\ndef check_grid(height, width, cell, patch):\n    if height % cell or width % cell or cell % patch:\n        raise ValueError(f'the cell ({cell} px) must divide the tile ({height} x {width}), and the patch ({patch} px) '\n                         'the cell')\n\n\ndef soil_values(x, y, step=4):\n    \"\"\"Bare-soil value per band, scaled to [0, 1] like the model input: the median over background pixels (label 0)\n    of uint16 tiles x (n, bands, H, W) with labels y (n, H, W), on every step-th pixel.\"\"\"\n    xs, bg = x[:, :, ::step, ::step], y[:, ::step, ::step] == 0\n    return np.array([np.median(xs[:, b][bg]) for b in range(x.shape[1])], dtype=np.float32) / 65535.0\n\n\ndef weed_prob(logits):\n    return torch.softmax(logits.float(), dim=1)[:, WEED]\n\n\ndef cell_scores(model, x, cell, amp=True):\n    \"\"\"Score s of every cell, (n, H / cell, W / cell), for a batch x (n, bands, H, W) scaled to [0, 1].\"\"\"\n    with autocast(x, amp):\n        logits = model(x)\n    return F.avg_pool2d(weed_prob(logits)[:, None], cell)[:, 0]\n\n\ndef cell_sum(p, rc, cell):\n    \"\"\"Sum of a map (..., H, W) over cell rc = (row, col).\"\"\"\n    r, c = rc\n    return p[..., r * cell:(r + 1) * cell, c * cell:(c + 1) * cell].sum((-2, -1))\n\n\ndef integrated_gradients(model, x, base, rc, cell, steps=32, batch=4, amp=True):\n    \"\"\"Attributions (bands, H, W) of the score of cell rc for one tile x (bands, H, W), from the bare-soil image, by\n    the midpoint rule over `steps` points; also s(x) - s(baseline), which the attributions should sum to.\"\"\"\n    base_img = base.view(-1, 1, 1).expand_as(x)\n    diff = x - base_img\n    grad_sum = torch.zeros_like(x)\n    alphas = (torch.arange(steps, device=x.device, dtype=torch.float32) + 0.5) / steps\n    for i in range(0, steps, batch):\n        xi = (base_img + alphas[i:i + batch].view(-1, 1, 1, 1) * diff).detach().requires_grad_(True)\n        with autocast(xi, amp):\n            logits = model(xi)\n        total = cell_sum(weed_prob(logits), rc, cell).sum()  # cell^2 * s per image, kept large for float16\n        grad_sum += torch.autograd.grad(total, xi)[0].float().sum(0)\n    with torch.no_grad():\n        s_x, s_base = cell_scores(model, torch.stack([x, base_img]), cell, amp)[:, rc[0], rc[1]].tolist()\n    return diff * grad_sum / (steps * cell * cell), s_x - s_base\n\n\ndef grad_cams(model, x, cells_rc, cell, amp=True):\n    \"\"\"Grad-CAM map (H, W) of the score of each cell in cells_rc, on decoder features computed once for the tile.\"\"\"\n    with torch.no_grad(), autocast(x, amp):\n        feat = model.features(x[None])\n    maps = []\n    for rc in cells_rc:\n        f = feat.float().detach().requires_grad_(True)\n        with autocast(x, amp):\n            logits = model.head(f, x.shape[-2:])\n        g = torch.autograd.grad(cell_sum(weed_prob(logits), rc, cell).sum(), f)[0].float() / (cell * cell)\n        cam = F.relu((g.mean((2, 3), keepdim=True) * f).sum(1, keepdim=True))\n        maps.append(F.interpolate(cam, size=x.shape[-2:], mode='bilinear', align_corners=False)[0, 0].detach())\n    return maps\n\n\ndef patch_sums(m, patch):\n    \"\"\"Sum of a map (H, W) over each patch: (H / patch, W / patch).\"\"\"\n    return F.avg_pool2d(m[None, None].float(), patch)[0, 0] * patch * patch\n\n\ndef top_mask(scores, n):\n    \"\"\"The n patches with the highest scores, as a boolean mask of the patch grid.\"\"\"\n    mask = torch.zeros(scores.numel(), dtype=torch.bool, device=scores.device)\n    mask[torch.topk(scores.flatten(), n).indices] = True\n    return mask.view_as(scores)\n\n\ndef random_mask(rng, shape, n, region=None):\n    \"\"\"n patches drawn without replacement, from the whole grid or from the patches where `region` is True.\"\"\"\n    cand = np.flatnonzero(np.ones(shape, bool) if region is None else region)\n    mask = np.zeros(int(np.prod(shape)), bool)\n    mask[rng.choice(cand, size=n, replace=False)] = True\n    return torch.from_numpy(mask.reshape(shape))\n\n\ndef occlude(x, base, mask, patch):\n    \"\"\"x (bands, H, W) with the patches of `mask` (H / patch, W / patch) replaced by the bare-soil values.\"\"\"\n    up = mask.to(x.device).repeat_interleave(patch, 0).repeat_interleave(patch, 1)\n    return torch.where(up[None], base.view(-1, 1, 1), x)\n\n\ndef cell_region(shape, rc, cell, patch):\n    \"\"\"Patches of the grid that lie inside cell rc.\"\"\"\n    k = cell // patch\n    region = np.zeros(shape, bool)\n    region[rc[0] * k:(rc[0] + 1) * k, rc[1] * k:(rc[1] + 1) * k] = True\n    return region\n\n\n@torch.no_grad()\ndef deletion_tile(model, x, base, tested, cell, patch, rng, fractions=FRACTIONS, draws=10, batch=8, amp=True):\n    \"\"\"Deletion test for the tested cells of one tile. tested: {cell index: ((row, col), {method: map (H, W)})}.\n    At each fraction, every tested cell is scored after removing the top patches of each of its maps, and random\n    patches inside it (`draws` times); random patches over the whole tile (`draws` times) are drawn once per tile,\n    since one occluded image gives every cell's score. One row per (cell, occluded image).\"\"\"\n    check_grid(x.shape[1], x.shape[2], cell, patch)\n    grid = (x.shape[1] // patch, x.shape[2] // patch)\n    ranked = {k: {name: patch_sums(m, patch) for name, m in maps.items()} for k, (_, maps) in tested.items()}\n    jobs = []  # (cell index, or None for an occlusion shared by every tested cell; method; fraction; draw; mask)\n    for f in fractions:\n        n = int(math.ceil(f * grid[0] * grid[1]))\n        jobs += [(None, 'random_tile', f, d, random_mask(rng, grid, n)) for d in range(draws)]\n        for k, (rc, _) in tested.items():\n            region = cell_region(grid, rc, cell, patch)\n            jobs += [(k, name, f, 0, top_mask(sc, n)) for name, sc in ranked[k].items()]\n            jobs += [(k, 'random_cell', f, d, random_mask(rng, grid, min(n, int(region.sum())), region))\n                     for d in range(draws)]\n    rows = []\n    for i in range(0, len(jobs), batch):\n        part = jobs[i:i + batch]\n        s = cell_scores(model, torch.stack([occlude(x, base, j[-1], patch) for j in part]), cell, amp).cpu()\n        for (k, name, f, d, _), sj in zip(part, s):\n            for kk in (tested if k is None else [k]):\n                rc = tested[kk][0]\n                rows.append({'cell': kk, 'method': name, 'fraction': f, 'draw': d, 's_after': float(sj[rc])})\n    return rows\n\n\n@torch.no_grad()\ndef band_ablation(model, x, base, cell, amp=True):\n    \"\"\"Drop in every cell's score when one band at a time is set to its bare-soil value over the whole tile:\n    (bands, H / cell, W / cell).\"\"\"\n    xb = x.repeat(len(base), 1, 1, 1)\n    for b in range(len(base)):\n        xb[b, b] = base[b]\n    s0 = cell_scores(model, x[None], cell, amp)[0]\n    return s0[None] - cell_scores(model, xb, cell, amp)\n\n\ndef explain_tile(model, x, base, cells, cell, patch, rng, steps=32, draws=10, ig_batch=4, batch=8, amp=True):\n    \"\"\"Explanations for the cells of one tile. x: tile (bands, H, W) in [0, 1] on the model's device; cells: rows of\n    the decision table for this tile (columns cell, decision). Returns one row per cell, the deletion rows of the\n    SPRAY and DEFER cells, maps {name: (n_cells, H / 4, W / 4)} in float32, and the time spent.\"\"\"\n    check_grid(x.shape[1], x.shape[2], cell, patch)\n    t0 = time.time()\n    n_cols = x.shape[2] // cell\n    with torch.no_grad():\n        with autocast(x, amp):\n            pred = weed_prob(model(x[None]))[0]\n        s_model = F.avg_pool2d(pred[None, None], cell)[0, 0]\n    ablation = band_ablation(model, x, base, cell, amp)\n    rcs = [divmod(int(k), n_cols) for k in cells['cell']]\n    cams = grad_cams(model, x, rcs, cell, amp)\n    cell_rows, ig_maps, cam_maps, tested = [], [], [], {}\n    for (_, c), rc, cam in zip(cells.iterrows(), rcs, cams):\n        attr, gap = integrated_gradients(model, x, base, rc, cell, steps, ig_batch, amp)\n        ig_map = attr.sum(0)\n        ig_maps.append(F.avg_pool2d(ig_map[None, None], MAP_POOL)[0, 0].cpu().numpy())\n        cam_maps.append(F.avg_pool2d(cam[None, None], MAP_POOL)[0, 0].cpu().numpy())\n        k = int(c['cell'])\n        cell_rows.append({'cell': k, 's_model': float(s_model[rc]), 'ig_gap': gap, 'ig_sum': float(ig_map.sum()),\n                          **{f'ig_{b}': v for b, v in zip(BANDS, attr.sum((1, 2)).tolist())},\n                          **{f'ablate_{b}': float(ablation[i][rc]) for i, b in enumerate(BANDS)}})\n        if c['decision'] in (SPRAY, DEFER):\n            in_cell = torch.zeros_like(pred)\n            in_cell[rc[0] * cell:(rc[0] + 1) * cell, rc[1] * cell:(rc[1] + 1) * cell] = 1\n            tested[k] = (rc, {'integrated_gradients': ig_map, 'grad_cam': cam, 'predicted_weed': pred * in_cell})\n    t1 = time.time()\n    dels = deletion_tile(model, x, base, tested, cell, patch, rng, draws=draws, batch=batch, amp=amp) if tested else []\n    maps = {'integrated_gradients': np.stack(ig_maps), 'grad_cam': np.stack(cam_maps)}\n    return cell_rows, dels, maps, {'explain_s': t1 - t0, 'deletion_s': time.time() - t1, 'tested': len(tested)}\n\n\ndef decisions_for_test(model, test, dev):\n    \"\"\"The model's decisions in the primary setting, from its probabilities on the test split (src.weeds.analyze).\"\"\"\n    _, probs = evaluate(model, test, dev, keep_probs=True)\n    cells = cell_table(test['ids'], to_3class(test['y']), probs[:, 1], PRIMARY['cell_size'])\n    return primary_decisions(cells, test_location_folds(list(test['ids'])))\n\n\ndef decision_check(t, expected):\n    \"\"\"Decision metrics and thresholds of table t next to those reported for the same model (a go_no_go.json).\"\"\"\n    m = decision_metrics(t['decision'].to_numpy(), t['w'].to_numpy() >= PRIMARY['tau'], t['w'].to_numpy(),\n                         t['crop'].to_numpy())\n    lam = t.groupby('fold')[['lam_lo', 'lam_hi']].first()\n    got = {'spray_reduction': m['spray_reduction'], 'deferred': m['deferred'],\n           'missed_need_rate': m['missed_need_rate'], 'false_spray_rate': m['false_spray_rate'],\n           'lam_lo': lam['lam_lo'].tolist(), 'lam_hi': lam['lam_hi'].tolist()}\n    if expected is None:\n        return {'computed': got}\n    want = {'spray_reduction': expected['checks']['C2_spray_reduction']['value'],\n            'deferred': expected['checks']['C3_deferred']['value'],\n            **{k: expected['info'][k] for k in ('missed_need_rate', 'false_spray_rate', 'lam_lo', 'lam_hi')}}\n    diff = max(float(np.max(np.abs(np.subtract(got[k], want[k])))) for k in want)\n    return {'computed': got, 'reported': want, 'max_abs_diff': diff}\n\n\ndef faithfulness(deletion, cells):\n    \"\"\"Per method and removed fraction: mean relative drop of s, share of cells where the drop beats the mean drop of\n    each random control, and the share of SPRAY cells that are no longer SPRAY. SPRAY is judged on the rescored s\n    (s_model) with the cell's thresholds, so that a cell sitting on a threshold does not count as flipped by\n    rounding alone.\"\"\"\n    if deletion.empty:  # no SPRAY or DEFER cell among those explained\n        return {}\n    df = deletion.merge(cells[['tile', 'cell', 's_model', 'lam_lo', 'lam_hi']], on=['tile', 'cell'])\n    df['spray_before'] = decide_conformal(df['s_model'], df['lam_lo'], df['lam_hi']) == SPRAY\n    df['drop'] = df['s_model'] - df['s_after']\n    df['rel_drop'] = df['drop'] / df['s_model']\n    df['flipped'] = decide_conformal(df['s_after'], df['lam_lo'], df['lam_hi']) != SPRAY\n    per_cell = df.groupby(['tile', 'cell', 'spray_before', 'method', 'fraction'], as_index=False)[\n        ['drop', 'rel_drop', 'flipped']].mean()\n    out = {}\n    for f, g in per_cell.groupby('fraction'):\n        rand = {ctrl: g[g['method'] == ctrl].set_index(['tile', 'cell'])['drop'] for ctrl in CONTROLS}\n        for m in METHODS + CONTROLS:\n            gm = g[g['method'] == m].set_index(['tile', 'cell'])\n            res = {'cells': len(gm), 'mean_rel_drop': float(gm['rel_drop'].mean()),\n                   'spray_flip_rate': float(gm[gm['spray_before']]['flipped'].mean())}\n            if m in METHODS:\n                for ctrl in CONTROLS:\n                    res[f'beats_{ctrl}'] = float((gm['drop'] > rand[ctrl].loc[gm.index]).mean())\n            out.setdefault(m, {})[str(f)] = res\n    return out\n\n\ndef run(args, dev):\n    out = Path(args.out)\n    parts = out / 'parts'\n    parts.mkdir(parents=True, exist_ok=True)\n    config = {'version': XAI_VERSION, 'ckpt': Path(args.ckpt).name,\n              **{k: getattr(args, k) for k in ('steps', 'patch', 'draws', 'seed', 'amp')}}\n    if (out / 'xai_config.json').exists() and json.loads((out / 'xai_config.json').read_text()) != config:\n        raise ValueError(f'{out} holds results made with other settings ({(out / \"xai_config.json\").read_text()}); '\n                         f'this run has {config}: use a new output folder')\n    (out / 'xai_config.json').write_text(json.dumps(config))\n\n    model = load_checkpoint(args.ckpt, len(BANDS), 3, dev)\n    for p in model.parameters():\n        p.requires_grad_(False)\n    if (out / 'soil_baseline.json').exists():\n        soil = np.array(json.loads((out / 'soil_baseline.json').read_text())['values'], dtype=np.float32)\n    else:\n        tr = load_npz(Path(args.data) / 'weedsgalore_train.npz')\n        soil = soil_values(tr['x'], tr['y'])\n        del tr\n        (out / 'soil_baseline.json').write_text(json.dumps({'bands': BANDS, 'values': soil.tolist(),\n                                                            'source': 'median of background pixels, train split'}))\n    base = torch.from_numpy(soil).to(dev)\n\n    test = load_npz(Path(args.data) / 'weedsgalore_test.npz')\n    if (out / 'decisions.csv').exists():  # fixed by the first session, so every tile explains the same decisions\n        cells = pd.read_csv(out / 'decisions.csv')\n    else:\n        cells = decisions_for_test(model, test, dev)\n        expected = json.loads(Path(args.expect).read_text()) if args.expect else None\n        check = decision_check(cells, expected)\n        (out / 'decision_check.json').write_text(json.dumps(clean(check), indent=1))\n        if check.get('max_abs_diff', 0) > 1e-6:\n            print(f'WARNING: decisions differ from the reported ones (max abs diff {check[\"max_abs_diff\"]:.2e}); '\n                  'see decision_check.json', flush=True)\n        cells.to_csv(out / 'decisions.csv', index=False)\n    counts = {name: int((cells['decision'] == k).sum()) for k, name in ((SPRAY, 'spray'), (NO_SPRAY, 'no_spray'),\n                                                                         (DEFER, 'defer'))}\n    print(f'{len(cells)} cells, decisions {counts}', flush=True)\n\n    tiles = list(dict.fromkeys(cells['tile']))[:args.limit_tiles or None]\n    if not all((parts / f'{t}.npz').exists() for t in tiles):\n        (out / 'xai_summary.json').unlink(missing_ok=True)  # only a finished run has a summary\n    index = {t: i for i, t in enumerate(test['ids'])}\n    t0 = time.time()\n    for k, tile in enumerate(tiles):\n        part = parts / f'{tile}.npz'\n        if part.exists():\n            continue\n        if time.time() - t0 > args.budget_hours * 3600:\n            print('time budget reached: rerun to resume', flush=True)\n            return\n        x = torch.from_numpy(test['x'][index[tile]].astype(np.float32) / 65535.0).to(dev)\n        tile_cells = cells[cells['tile'] == tile]\n        rows, dels, maps, timing = explain_tile(model, x, base, tile_cells, PRIMARY['cell_size'], args.patch,\n                                                np.random.default_rng([args.seed, k]), args.steps, args.draws,\n                                                args.ig_batch, args.batch, args.amp)\n        for r, (_, c) in zip(rows, tile_cells.iterrows()):\n            r['decision'] = int(c['decision'])  # kept with the part, checked when merging\n        tmp = parts / f'{tile}.tmp.npz'\n        np.savez_compressed(tmp, cells=json.dumps(rows), deletion=json.dumps(dels), **maps)\n        tmp.replace(part)\n        err = max(abs(r['ig_sum'] - r['ig_gap']) for r in rows)\n        print(f'{tile}: {len(rows)} cells ({timing[\"tested\"]} tested) in {timing[\"explain_s\"]:.0f} s + deletion '\n              f'{timing[\"deletion_s\"]:.0f} s | integrated gradients completeness: max abs error {err:.2e}', flush=True)\n        if args.limit_tiles:\n            tested_all = int(cells['decision'].isin([SPRAY, DEFER]).sum())\n            per_tested = timing['deletion_s'] / max(timing['tested'], 1)\n            est = timing['explain_s'] * cells['tile'].nunique() + per_tested * tested_all\n            print(f'estimated full run: {est / 60:.0f} min ({cells[\"tile\"].nunique()} tiles, {tested_all} tested '\n                  'cells)', flush=True)\n\n    cell_rows, del_rows, maps = [], [], {m: [] for m in ('integrated_gradients', 'grad_cam')}\n    for tile in tiles:\n        with np.load(parts / f'{tile}.npz') as p:\n            cell_rows += [{'tile': tile, **r} for r in json.loads(str(p['cells']))]\n            del_rows += [{'tile': tile, **r} for r in json.loads(str(p['deletion']))]\n            for m in maps:\n                maps[m].append(p[m])\n    merged = pd.DataFrame(cell_rows).rename(columns={'decision': 'decision_part'})\n    xai = cells.merge(merged, on=['tile', 'cell'])\n    if (xai['decision'] != xai['decision_part']).any():\n        raise ValueError('parts were computed for other decisions than decisions.csv: use a new output folder')\n    xai = xai.drop(columns='decision_part')\n    deletion = pd.DataFrame(del_rows)\n    xai.to_csv(out / 'xai_cells.csv', index=False)\n    deletion.to_csv(out / 'xai_deletion.csv', index=False)\n    # maps are stacked in the order of cell_rows: one map per (tile, cell)\n    np.savez_compressed(out / 'xai_maps.npz', tile=np.array([r['tile'] for r in cell_rows]),\n                        cell=np.array([r['cell'] for r in cell_rows]), **{m: np.concatenate(v) for m, v in maps.items()})\n    err = (xai['ig_sum'] - xai['ig_gap']).abs()\n    rescored = decide_conformal(xai['s_model'], xai['lam_lo'], xai['lam_hi'])\n    summary = {'config': config, 'cells': len(xai), 'decisions': counts,\n               'decision_check': json.loads((out / 'decision_check.json').read_text())\n               if (out / 'decision_check.json').exists() else None,\n               'rescored_decision_differs': int((rescored != xai['decision']).sum()),\n               'score_check_max_abs_diff': float((xai['s_model'] - xai['s']).abs().max()),\n               'soil_baseline': dict(zip(BANDS, soil.tolist())),\n               'ig_completeness_abs_error': {'median': float(err.median()), 'max': float(err.max())},\n               'ig_completeness_rel_error_median': float((err / xai['ig_gap'].abs().clip(lower=1e-6)).median()),\n               'faithfulness': faithfulness(deletion, xai)}\n    (out / 'xai_summary.json').write_text(json.dumps(clean(summary), indent=1))\n    print(json.dumps(clean({k: summary[k] for k in ('decisions', 'rescored_decision_differs',\n                                                    'ig_completeness_abs_error')}), indent=1))\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument('--data', required=True)\n    ap.add_argument('--ckpt', required=True)\n    ap.add_argument('--out', required=True)\n    ap.add_argument('--expect', default=None, help='go_no_go.json reported for this model, to check the decisions')\n    ap.add_argument('--steps', type=int, default=32, help='integrated gradients: points on the path')\n    ap.add_argument('--patch', type=int, default=20, help='deletion test: patch size in pixels (5 cm)')\n    ap.add_argument('--draws', type=int, default=10, help='deletion test: random draws per control and fraction')\n    ap.add_argument('--ig-batch', type=int, default=4)\n    ap.add_argument('--batch', type=int, default=8)\n    ap.add_argument('--seed', type=int, default=0)\n    ap.add_argument('--no-amp', dest='amp', action='store_false', help='float32 throughout (slower, more memory)')\n    ap.add_argument('--budget-hours', type=float, default=3.0)\n    ap.add_argument('--limit-tiles', type=int, default=0, help='explain only the first N tiles (smoke test)')\n    args = ap.parse_args()\n    torch.manual_seed(args.seed)\n    dev = torch.device('cuda' if torch.cuda.is_available() else 'cpu')\n    print('device:', dev, torch.cuda.get_device_name(0) if dev.type == 'cuda' else '')\n    run(args, dev)\n\n\nif __name__ == '__main__':\n    main()\n", "src/weeds/counterfactual.py": "\"\"\"Decision-level counterfactual explanations of the spraying decisions, computed from the P(weed) map alone.\n\nA cell's decision is a threshold rule on its score s = mean P(weed) over the cell (src.weeds.decision), so it can be\nexplained by what would have to change for the decision to change (counterfactual explanations, Wachter et al. 2018,\nHarvard Journal of Law & Technology):\n  SPRAY      the fewest patches of the cell's fixed 20 x 20 px grid (5 cm x 5 cm) whose removal ends SPRAY, and gives\n             NO_SPRAY;\n  DEFER      the predicted weed area to add for SPRAY, and the patches / area to remove for NO_SPRAY;\n  NO_SPRAY   the predicted weed area to add for SPRAY.\nOnly patches inside the cell are removed (the model also sees context outside it). Removal takes the patches with the\nmost predicted weed first and assumes, without running the model again, that a removed patch becomes weed-free and\nthe rest of the cell is unchanged: s drops by the patch's P(weed) mass / cell^2. The decision is monotone in s, so\nunder that assumption largest-first gives the fewest patches. The deletion test of src.weeds.explain removes the same\npatches (its 'predicted_weed' ranking) and rescores the image with the model, which measures how far the estimate is\noff (compare_with_deletion), at its fixed removal sizes only. Adding weed follows from the rule alone and cannot be\nchecked with the model.\nAreas are areas of predicted weed: P(weed) summed over pixels, times the pixel area.\n\n  python -m src.weeds.counterfactual --probs <probs_test.npz> --decisions <decisions.csv> --out <dir>\n                                     [--deletion <xai_deletion.csv>]\n\"\"\"\nimport argparse\nimport json\nimport math\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\n\nfrom src.weeds.analyze import PRIMARY\nfrom src.weeds.decision import DEFER, NO_SPRAY, SPRAY, decide_conformal\n\nGSD_CM = 0.25  # ground sampling distance of WeedsGalore: 2.5 mm per pixel (DJI P4 Multispectral at 5 m)\nPATCH = 20  # 20 px = 5 cm, the patch of the deletion test in src.weeds.explain\nSCORE_TOL = 1e-5  # the decisions table and the probability map may come from two GPU runs of the same model\nSPRAYNESS = {NO_SPRAY: 0, DEFER: 1, SPRAY: 2}\n\n\ndef ranked_patches(p_cell, patch):\n    \"\"\"Patches of one cell's P(weed) map (cell, cell) ranked by their P(weed) mass, largest first: their indices in\n    row-major order of the cell's patch grid, and their masses (float64).\"\"\"\n    c = p_cell.shape[0]\n    if p_cell.shape != (c, c) or c % patch:\n        raise ValueError(f'the cell map must be square with a side divisible by the patch ({patch} px): {p_cell.shape}')\n    k = c // patch\n    m = p_cell.astype(np.float64).reshape(k, patch, k, patch).sum((1, 3)).ravel()\n    order = np.argsort(-m, kind='stable')\n    return order, m[order]\n\n\ndef removal_path(s, masses, cell):\n    \"\"\"Estimated score after removing the first 0, 1, ..., all ranked patches.\"\"\"\n    return s - np.concatenate([[0.0], np.cumsum(masses)]) / cell ** 2\n\n\ndef first_index(mask):\n    i = np.flatnonzero(mask)\n    return float(i[0]) if len(i) else np.nan\n\n\ndef _cell_map(p_weed, row, cell, cell_idx):\n    cols = p_weed.shape[2] // cell\n    r, c = divmod(int(cell_idx), cols)\n    return r, c, p_weed[row, r * cell:(r + 1) * cell, c * cell:(c + 1) * cell]\n\n\ndef _check_inputs(ids, p_weed, decisions, cell, patch):\n    n, h, w = p_weed.shape\n    if len(ids) != n:\n        raise ValueError(f'{len(ids)} ids for {n} probability maps')\n    if h % cell or w % cell or cell % patch:\n        raise ValueError(f'the cell ({cell} px) must divide the tile ({h} x {w}), and the patch ({patch} px) the cell')\n    missing = set(decisions['tile']) - set(ids)\n    if missing:\n        raise ValueError(f'no probability map for tiles {sorted(missing)}')\n\n\ndef counterfactual_table(ids, p_weed, decisions, cell, patch=PATCH, gsd_cm=GSD_CM, score_tol=SCORE_TOL):\n    \"\"\"One row per cell of the decision table (columns tile, date, s, lam_lo, lam_hi, decision, and cell, the cell's\n    row-major index in its tile; when absent, the row order within each tile). p_weed: (n, H, W) maps of the tiles ids.\n    Areas in cm^2 of predicted weed; k_* in patches (NaN when no removal reaches the decision); `patches` lists the\n    ranked patches (row-major indices in the tile's patch grid) up to the larger finite k, so that the first k_not_spray\n    of them end SPRAY. Increases are infima: where the rule's inequality is strict, an arbitrarily small extra is\n    needed.\"\"\"\n    ids = list(ids)\n    _check_inputs(ids, p_weed, decisions, cell, patch)\n    t = decisions.copy()\n    if 'cell' not in t:\n        t['cell'] = t.groupby('tile').cumcount()\n    row_of = {tile: i for i, tile in enumerate(ids)}\n    cell_cm2 = cell ** 2 * gsd_cm ** 2\n    k, grid_cols = cell // patch, p_weed.shape[2] // patch\n    rows = []\n    for _, d in t.iterrows():\n        r, c, p_cell = _cell_map(p_weed, row_of[d['tile']], cell, d['cell'])\n        order, masses = ranked_patches(p_cell, patch)\n        s = masses.sum() / cell ** 2\n        if abs(s - d['s']) > score_tol:\n            raise ValueError(f'{d[\"tile\"]} cell {d[\"cell\"]}: score {s:.6f} from the map, {d[\"s\"]:.6f} in the table')\n        lo, hi = float(d['lam_lo']), float(d['lam_hi'])\n        path = removal_path(s, masses, cell)\n        dec = decide_conformal(path, lo, hi)\n        if dec[0] != d['decision']:\n            raise ValueError(f'{d[\"tile\"]} cell {d[\"cell\"]}: decision {dec[0]} from the map, {d[\"decision\"]} in the '\n                             'table')\n        k_not, k_no = first_index(dec != SPRAY), first_index(dec == NO_SPRAY)\n\n        def removed_cm2(kk):\n            return (s - path[int(kk)]) * cell_cm2 if np.isfinite(kk) else np.nan\n\n        upto = int(np.nanmax([k_not, k_no, 0.0]))\n        tile_patch = (r * k + order // k) * grid_cols + c * k + order % k\n        rows.append({'tile': d['tile'], 'cell': int(d['cell']), 'date': d['date'], 'decision': int(d['decision']),\n                     's': s, 'lam_lo': lo, 'lam_hi': hi, 'weed_cm2': s * cell_cm2,\n                     'add_to_spray_cm2': max(0.0, max(lo, hi) - s) * cell_cm2,\n                     'remove_to_no_spray_cm2': max(0.0, s - min(lo, hi)) * cell_cm2,\n                     'k_not_spray': k_not, 'removed_cm2_not_spray': removed_cm2(k_not),\n                     'k_no_spray': k_no, 'removed_cm2_no_spray': removed_cm2(k_no),\n                     'patches': ' '.join(str(int(i)) for i in tile_patch[:upto])})\n    return pd.DataFrame(rows)\n\n\ndef compare_with_deletion(table, ids, p_weed, deletion, cell, patch=PATCH, gsd_cm=GSD_CM):\n    \"\"\"The estimated score after removing the top-n patches next to the model's rescoring in the deletion test of\n    src.weeds.explain, whose 'predicted_weed' ranking removes the same patches (n = ceil(fraction x patches per\n    tile)). One row per tested cell and fraction.\"\"\"\n    ids = list(ids)\n    row_of = {tile: i for i, tile in enumerate(ids)}\n    grid = (p_weed.shape[1] // patch, p_weed.shape[2] // patch)\n    cell_cm2 = cell ** 2 * gsd_cm ** 2\n    t = table.set_index(['tile', 'cell'])\n    paths, rows = {}, []\n    for _, d in deletion[deletion['method'] == 'predicted_weed'].iterrows():\n        key = (d['tile'], int(d['cell']))\n        c = t.loc[key]\n        if key not in paths:\n            _, _, p_cell = _cell_map(p_weed, row_of[key[0]], cell, key[1])\n            masses = ranked_patches(p_cell, patch)[1]\n            paths[key] = removal_path(masses.sum() / cell ** 2, masses, cell)\n        path = paths[key]\n        n = int(math.ceil(d['fraction'] * grid[0] * grid[1]))  # the same float expression as explain.deletion_tile\n        s_hat = path[min(n, len(path) - 1)]\n        rows.append({'tile': key[0], 'cell': key[1], 'date': c['date'], 'decision': int(c['decision']),\n                     'fraction': d['fraction'], 'n_patches': n, 's': c['s'], 's_after_model': d['s_after'],\n                     's_after_estimate': s_hat, 'lam_lo': c['lam_lo'], 'lam_hi': c['lam_hi']})\n    out = pd.DataFrame(rows)\n    out['error_cm2'] = (out['s_after_estimate'] - out['s_after_model']) * cell_cm2  # < 0: estimate drops more\n    out['decision_model'] = decide_conformal(out['s_after_model'], out['lam_lo'], out['lam_hi'])\n    out['decision_estimate'] = decide_conformal(out['s_after_estimate'], out['lam_lo'], out['lam_hi'])\n    return out\n\n\ndef _q(x):\n    x = pd.Series(x, dtype=float)\n    finite = x[np.isfinite(x)]\n    res = {'n': len(x), 'not_reachable': int(x.isna().sum() + np.isinf(x).sum())}\n    if len(finite):\n        res.update({k: float(v) for k, v in zip(('min', 'p25', 'median', 'p75', 'max'),\n                                                finite.quantile([0, .25, .5, .75, 1]))})\n    return res\n\n\ndef summarize(table):\n    sp, de, ns = (table[table['decision'] == v] for v in (SPRAY, DEFER, NO_SPRAY))\n    out = {'cells': len(table), 'decisions': {'spray': len(sp), 'no_spray': len(ns), 'defer': len(de)},\n           'spray': {'k_not_spray': _q(sp['k_not_spray']), 'removed_cm2_not_spray': _q(sp['removed_cm2_not_spray']),\n                     'k_no_spray': _q(sp['k_no_spray']),\n                     **{f'k_not_spray_le_{k}': int((sp['k_not_spray'] <= k).sum()) for k in (1, 2, 4)}},\n           'defer': {'add_to_spray_cm2': _q(de['add_to_spray_cm2']),\n                     'remove_to_no_spray_cm2': _q(de['remove_to_no_spray_cm2']), 'k_no_spray': _q(de['k_no_spray'])},\n           'no_spray': {'add_to_spray_cm2': _q(ns['add_to_spray_cm2'])},\n           'by_date': {}}\n    for date, g in table.groupby('date'):\n        gs, gn = g[g['decision'] == SPRAY], g[g['decision'] == NO_SPRAY]\n        out['by_date'][date] = {'spray': len(gs), 'spray_k_not_spray_median': float(gs['k_not_spray'].median())\n                                if len(gs) else None,\n                                'no_spray': len(gn), 'no_spray_add_to_spray_cm2_median':\n                                    float(gn['add_to_spray_cm2'].median()) if len(gn) else None}\n    return out\n\n\ndef summarize_comparison(cmp):\n    lenient = cmp['decision_estimate'].map(SPRAYNESS) < cmp['decision_model'].map(SPRAYNESS)\n    was_spray = cmp['decision'] == SPRAY\n    drop_ratio = (cmp['s'] - cmp['s_after_estimate']) / (cmp['s'] - cmp['s_after_model'])\n    by_fraction = {}\n    for f, g in cmp.groupby('fraction'):\n        e = g['error_cm2'].abs()\n        by_fraction[str(f)] = {'rows': len(g), 'abs_error_cm2_median': float(e.median()),\n                               'abs_error_cm2_p95': float(e.quantile(.95)), 'abs_error_cm2_max': float(e.max()),\n                               'drop_ratio_median': float(drop_ratio[g.index].median())}\n    agree = cmp['decision_model'] == cmp['decision_estimate']\n    # rows where neither changes the decision agree trivially; agreement where at least one of them changes it\n    changed = (cmp['decision_model'] != cmp['decision']) | (cmp['decision_estimate'] != cmp['decision'])\n    return {'rows': len(cmp), 'cells': int(cmp[['tile', 'cell']].drop_duplicates().shape[0]),\n            'decision_agreement': float(agree.mean()), 'disagreements': int((~agree).sum()),\n            'rows_decision_changed': int(changed.sum()),\n            'decision_agreement_where_changed': float(agree[changed].mean()) if changed.any() else None,\n            'disagreements_estimate_less_spray': int((~agree & lenient).sum()),\n            'spray_end_agreement': float(((cmp['decision_model'] != SPRAY) == (cmp['decision_estimate'] != SPRAY))\n                                         [was_spray].mean()),\n            'spray_rows': int(was_spray.sum()), 'by_fraction': by_fraction}\n\n\ndef main():\n    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)\n    ap.add_argument('--probs', required=True, help='probs_test.npz of the model that made the decisions')\n    ap.add_argument('--decisions', required=True, help='decisions.csv (src.weeds.explain) of the same model')\n    ap.add_argument('--deletion', help='xai_deletion.csv of the same model, to check the estimate against the model')\n    ap.add_argument('--out', required=True)\n    ap.add_argument('--cell', type=int, default=PRIMARY['cell_size'])\n    ap.add_argument('--patch', type=int, default=PATCH)\n    args = ap.parse_args()\n    with np.load(args.probs, allow_pickle=False) as z:\n        ids, p_weed = list(z['ids']), z['p_weed']\n    decisions = pd.read_csv(args.decisions)\n    t0 = time.perf_counter()\n    table = counterfactual_table(ids, p_weed, decisions, args.cell, args.patch)\n    seconds = time.perf_counter() - t0\n    summary = {'inputs': {'probs': args.probs, 'decisions': args.decisions, 'deletion': args.deletion,\n                          'cell_px': args.cell, 'patch_px': args.patch, 'gsd_cm': GSD_CM},\n               **summarize(table)}\n    if args.deletion:\n        summary['check_against_model'] = summarize_comparison(\n            compare_with_deletion(table, ids, p_weed, pd.read_csv(args.deletion), args.cell, args.patch))\n    out = Path(args.out)\n    out.mkdir(parents=True, exist_ok=True)\n    table.to_csv(out / 'counterfactual_cells.csv', index=False)\n    (out / 'counterfactual_summary.json').write_text(json.dumps(summary, indent=1))\n    print(json.dumps(summary, indent=1))\n    print(f'{len(table)} cells in {seconds:.2f} s ({1000 * seconds / len(table):.2f} ms per cell, this machine)')\n\n\nif __name__ == '__main__':\n    main()\n", "src/weeds/counterfactual_check.py": "\"\"\"Model check of the counterfactual explanations of src.weeds.counterfactual.\n\nsrc.weeds.counterfactual estimates from the P(weed) map alone, without running the model again, the fewest 20 x 20 px\npatches of a cell whose removal ends SPRAY (k_not_spray) and gives NO_SPRAY (k_no_spray). Here the model rescores the\ntile after the first k = 1, 2, ... ranked patches of the cell are replaced by bare soil (the values of\nsrc.weeds.explain), for every SPRAY and DEFER cell. Patches are ranked by the model's own P(weed) mass inside the cell,\nas the 'predicted_weed' ranking of the deletion test, i.e. in the order the explanation names them. The scan stops\nonce the model gives NO_SPRAY and k has passed it and both estimates by `extra` patches, or when every patch of the\ncell is gone. The first k at which the model's decision changes checks the explanation as it is shown (\"remove these\nk patches and the cell is no longer sprayed\"). It is not the fewest patches for the model: with context, another set\nof patches could change the decision with fewer. The scan also shows whether the decision changes back at a larger k,\nsince the model's score need not fall monotonically. Run on Colab GPU:\n\n  python -m src.weeds.counterfactual_check --data <dir with weedsgalore_test.npz> --ckpt <dlv3p_msi_3.pth>\n      --decisions <decisions.csv of src.weeds.explain> --soil <soil_baseline.json of src.weeds.explain> --out <dir>\n\nPer-tile results go to <out>/parts/, so an interrupted run resumes; settings are fixed per output folder.\n\"\"\"\nimport argparse\nimport hashlib\nimport json\nimport math\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport torch\nimport torch.nn.functional as F\n\nfrom src.weeds.analyze import PRIMARY, clean\nfrom src.weeds.authors import load_checkpoint\nfrom src.weeds.counterfactual import GSD_CM, PATCH, first_index, ranked_patches, removal_path\nfrom src.weeds.data import BANDS, load_npz\nfrom src.weeds.decision import DEFER, NO_SPRAY, SPRAY, decide_conformal\nfrom src.weeds.explain import autocast, cell_scores, check_grid, occlude, weed_prob\n\nCHECK_VERSION = 1  # bump when the scan changes: parts of another version are not merged\n\n\ndef weed_map(model, x, amp=True):\n    \"\"\"The model's P(weed) map (H, W) of one tile x (bands, H, W), float32.\"\"\"\n    with torch.no_grad(), autocast(x, amp):\n        return weed_prob(model(x[None]))[0]\n\n\ndef cell_ranking(pred, rc, cell, patch):\n    \"\"\"For cell rc = (row, col) of a P(weed) map: its patches ranked by P(weed) mass, largest first, as row-major\n    indices in the tile's patch grid; the cell's score s; and the estimated scores after removing 0, 1, ... of them.\"\"\"\n    r, c = rc\n    order, masses = ranked_patches(pred[r * cell:(r + 1) * cell, c * cell:(c + 1) * cell].double().cpu().numpy(),\n                                   patch)\n    side, grid_cols = cell // patch, pred.shape[1] // patch\n    s = float(pred[r * cell:(r + 1) * cell, c * cell:(c + 1) * cell].double().mean())\n    return (r * side + order // side) * grid_cols + c * side + order % side, s, removal_path(s, masses, cell)\n\n\ndef estimated_k(est, lam_lo, lam_hi):\n    d = decide_conformal(est, lam_lo, lam_hi)\n    return first_index(d != SPRAY), first_index(d == NO_SPRAY)\n\n\ndef scan_target(no_spray_at, k_estimates, extra):\n    \"\"\"How far the scan must go once the model has given NO_SPRAY at k = no_spray_at.\"\"\"\n    return max([no_spray_at] + [int(k) for k in k_estimates if np.isfinite(k)]) + extra\n\n\n@torch.no_grad()\ndef verify_tile(model, x, base, cells, cell, patch=PATCH, batch=8, extra=2, amp=True):\n    \"\"\"The scan for some cells of one tile: decision-table rows with columns cell, decision, lam_lo, lam_hi. x: the\n    tile (bands, H, W) in [0, 1] on the model's device; base: bare-soil value per band. Returns one row per scored\n    (cell, k), one row per cell, and the number of forward passes.\"\"\"\n    check_grid(x.shape[1], x.shape[2], cell, patch)\n    pred = weed_map(model, x, amp)\n    n_cols, grid = x.shape[2] // cell, (x.shape[1] // patch, x.shape[2] // patch)\n    passes, step_rows, cell_rows = 1, [], []\n    for _, c in cells.iterrows():\n        lo, hi = float(c['lam_lo']), float(c['lam_hi'])\n        rc = divmod(int(c['cell']), n_cols)\n        patches, s0, est = cell_ranking(pred, rc, cell, patch)\n        k_est = estimated_k(est, lo, hi)\n        scores = [s0]\n        no_at = 0 if decide_conformal(np.array([s0]), lo, hi)[0] == NO_SPRAY else None\n        k = 0\n        while k < len(patches) and (no_at is None or k < scan_target(no_at, k_est, extra)):\n            ks = range(k + 1, min(len(patches), k + batch) + 1)\n            imgs = []\n            for kk in ks:\n                mask = torch.zeros(grid[0] * grid[1], dtype=torch.bool)\n                mask[torch.from_numpy(patches[:kk].astype(np.int64))] = True\n                imgs.append(occlude(x, base, mask.view(grid), patch))\n            s = cell_scores(model, torch.stack(imgs), cell, amp)[:, rc[0], rc[1]].tolist()\n            passes += len(imgs)\n            for kk, sv in zip(ks, s):\n                scores.append(sv)\n                if no_at is None and decide_conformal(np.array([sv]), lo, hi)[0] == NO_SPRAY:\n                    no_at = kk\n            k = ks[-1]\n        scores = np.array(scores)\n        dec_model = decide_conformal(scores, lo, hi)\n        dec_est = decide_conformal(est[:len(scores)], lo, hi)\n        for kk in range(len(scores)):\n            step_rows.append({'cell': int(c['cell']), 'k': kk, 's_model': float(scores[kk]),\n                              's_estimate': float(est[kk]), 'decision_model': int(dec_model[kk]),\n                              'decision_estimate': int(dec_est[kk])})\n        k_not, k_no = first_index(dec_model != SPRAY), first_index(dec_model == NO_SPRAY)\n        cell_rows.append({\n            'cell': int(c['cell']), 'decision': int(c['decision']), 'decision_rescored': int(dec_model[0]),\n            's_model': s0, 'lam_lo': lo, 'lam_hi': hi,\n            'k_not_spray_estimate': k_est[0], 'k_no_spray_estimate': k_est[1],\n            'k_not_spray_model': k_not, 'k_no_spray_model': k_no, 'k_scanned': len(scores) - 1,\n            'reverts_to_spray': bool(np.isfinite(k_not) and (dec_model[int(k_not):] == SPRAY).any()),\n            'reverts_from_no_spray': bool(np.isfinite(k_no) and (dec_model[int(k_no):] != NO_SPRAY).any()),\n            'max_score_increase': float(max(np.diff(scores).max(initial=0.0), 0.0)),\n            'patches': ' '.join(str(int(p)) for p in patches[:len(scores) - 1])})\n    return step_rows, cell_rows, passes\n\n\ndef planned_passes(model, tiles, cells, cell, patch, batch, extra, amp=True):\n    \"\"\"Forward passes the scan needs if the model agrees with the estimate: one pass per tile for its P(weed) map,\n    then per cell the batches up to the estimated k plus `extra` (every patch when the estimate never gives NO_SPRAY).\n    More passes are needed where the model needs more patches than the estimate, fewer where the estimate never\n    gives NO_SPRAY but the model does.\n    tiles: {tile id: tile tensor on the model's device}; cells: decision-table rows of those tiles.\"\"\"\n    total = 0\n    for tile, x in tiles.items():\n        pred = weed_map(model, x, amp)\n        total += 1\n        for _, c in cells[cells['tile'] == tile].iterrows():\n            patches, _, est = cell_ranking(pred, divmod(int(c['cell']), x.shape[2] // cell), cell, patch)\n            k_est = estimated_k(est, float(c['lam_lo']), float(c['lam_hi']))\n            finite = [int(k) for k in k_est if np.isfinite(k)]\n            target = min(len(patches), max(finite) + extra if finite else len(patches))\n            total += min(len(patches), int(math.ceil(target / batch)) * batch)\n    return total\n\n\ndef _compare(model_k, est_k):\n    ok = model_k.notna() & est_k.notna()\n    d = (model_k - est_k)[ok].astype(int)\n    return {'cells': int(len(model_k)), 'compared': int(ok.sum()), 'model_not_reached': int(model_k.isna().sum()),\n            'estimate_not_reached': int(est_k.isna().sum()), 'exact': int((d == 0).sum()),\n            'estimate_too_small': int((d > 0).sum()), 'estimate_too_large': int((d < 0).sum()),\n            'model_minus_estimate': {str(v): int(n) for v, n in d.value_counts().sort_index().items()}}\n\n\ndef summarize_check(cells, steps, cell, gsd_cm=GSD_CM):\n    \"\"\"cells: one row per checked cell; steps: one row per (tile, cell, k) scored (k = 0 is the unoccluded tile).\"\"\"\n    cell_cm2 = cell ** 2 * gsd_cm ** 2\n    sp = cells[cells['decision_rescored'] == SPRAY]\n    tested = cells[cells['decision_rescored'].isin([SPRAY, DEFER])]\n    fragile = sp[sp['k_not_spray_estimate'] <= 2]\n    st = steps.merge(cells[['tile', 'cell', 'decision_rescored']], on=['tile', 'cell'])\n    st = st[st['k'] > 0]\n    err = (st['s_estimate'] - st['s_model']) * cell_cm2  # < 0: the estimate falls further than the model\n    agree = st['decision_model'] == st['decision_estimate']\n    changed = (st['decision_model'] != st['decision_rescored']) | (st['decision_estimate'] != st['decision_rescored'])\n    inc = cells['max_score_increase'] * cell_cm2\n    return {'cells': len(cells),\n            'decisions': {name: int((cells['decision'] == v).sum()) for v, name in ((SPRAY, 'spray'),\n                                                                                    (DEFER, 'defer'))},\n            'rescored_decision_differs': int((cells['decision'] != cells['decision_rescored']).sum()),\n            'k_not_spray': _compare(sp['k_not_spray_model'], sp['k_not_spray_estimate']),\n            'k_not_spray_fragile': _compare(fragile['k_not_spray_model'], fragile['k_not_spray_estimate']),\n            'k_no_spray': _compare(tested['k_no_spray_model'], tested['k_no_spray_estimate']),\n            'reverts_to_spray': int(cells['reverts_to_spray'].sum()),\n            'reverts_from_no_spray': int(cells['reverts_from_no_spray'].sum()),\n            'max_score_increase_cm2': {'median': float(inc.median()), 'max': float(inc.max())},\n            'curve': {'rows': len(st), 'error_cm2_median': float(err.median()),\n                      'abs_error_cm2_median': float(err.abs().median()),\n                      'abs_error_cm2_p95': float(err.abs().quantile(.95)), 'abs_error_cm2_max': float(err.abs().max()),\n                      'decision_agreement': float(agree.mean()), 'rows_decision_changed': int(changed.sum()),\n                      'decision_agreement_where_changed': float(agree[changed].mean()) if changed.any() else None}}\n\n\ndef run(args, dev):\n    out = Path(args.out)\n    parts = out / 'parts'\n    parts.mkdir(parents=True, exist_ok=True)\n    soil = json.loads(Path(args.soil).read_text())\n    if list(soil['bands']) != list(BANDS):\n        raise ValueError(f'soil values are for bands {soil[\"bands\"]}, the model takes {BANDS}')\n    # the inputs are part of the settings, so that parts made for other decisions or soil values are never merged\n    config = {'version': CHECK_VERSION, 'ckpt': Path(args.ckpt).name, 'cell': args.cell, 'patch': args.patch,\n              'extra': args.extra, 'amp': args.amp, 'soil': [float(v) for v in soil['values']],\n              'decisions_sha256': hashlib.sha256(Path(args.decisions).read_bytes()).hexdigest()}\n    cfg = out / 'cfcheck_config.json'\n    if cfg.exists() and json.loads(cfg.read_text()) != config:\n        raise ValueError(f'{out} holds results made with other settings ({cfg.read_text()}); this run has {config}: '\n                         'use a new output folder')\n    cfg.write_text(json.dumps(config))\n    base = torch.tensor(soil['values'], dtype=torch.float32, device=dev)\n    decisions = pd.read_csv(args.decisions)\n    todo = decisions[decisions['decision'].isin([SPRAY, DEFER])]\n    model = load_checkpoint(args.ckpt, len(BANDS), 3, dev)\n    for p in model.parameters():\n        p.requires_grad_(False)\n    test = load_npz(Path(args.data) / 'weedsgalore_test.npz')\n    index = {t: i for i, t in enumerate(test['ids'])}\n\n    def tile_tensor(t):\n        return torch.from_numpy(test['x'][index[t]].astype(np.float32) / 65535.0).to(dev)\n\n    all_tiles = list(dict.fromkeys(todo['tile']))\n    tiles = all_tiles[:args.limit_tiles or None]\n    cell = args.cell\n    t0, done_passes, done_s = time.time(), 0, 0.0\n    for tile in tiles:\n        part = parts / f'{tile}.json'\n        if part.exists():\n            continue\n        if time.time() - t0 > args.budget_hours * 3600:\n            print('time budget reached: rerun to resume', flush=True)\n            return\n        t1 = time.time()\n        steps, cells, passes = verify_tile(model, tile_tensor(tile), base, todo[todo['tile'] == tile], cell,\n                                           args.patch, args.batch, args.extra, args.amp)\n        if dev.type == 'cuda':\n            torch.cuda.synchronize()\n        sec = time.time() - t1\n        done_passes, done_s = done_passes + passes, done_s + sec\n        tmp = part.with_suffix('.tmp')\n        tmp.write_text(json.dumps(clean({'steps': steps, 'cells': cells, 'passes': passes, 'seconds': sec})))\n        tmp.replace(part)\n        print(f'{tile}: {len(cells)} cells, {passes} forward passes in {sec:.0f} s', flush=True)\n    if args.limit_tiles:\n        if done_passes:\n            planned = planned_passes(model, {t: tile_tensor(t) for t in all_tiles}, todo, cell, args.patch,\n                                     args.batch, args.extra, args.amp)\n            print(f'estimated full run: about {planned * done_s / done_passes / 60:.0f} min ({planned} forward '\n                  f'passes at {done_s / done_passes:.3f} s each; more if the model needs more patches than the '\n                  'estimate)', flush=True)\n        return\n    if not all((parts / f'{t}.json').exists() for t in all_tiles):\n        return\n    steps, cells, passes, seconds = [], [], 0, 0.0\n    for tile in all_tiles:\n        d = json.loads((parts / f'{tile}.json').read_text())\n        steps += [{'tile': tile, **r} for r in d['steps']]\n        cells += [{'tile': tile, **r} for r in d['cells']]\n        passes, seconds = passes + d['passes'], seconds + d['seconds']\n    steps = pd.DataFrame(steps)\n    cells = pd.DataFrame(cells).merge(todo[['tile', 'cell', 'date']], on=['tile', 'cell'])\n    for col in [c for c in cells if c.startswith('k_') and c.endswith(('_estimate', '_model'))]:\n        cells[col] = cells[col].astype(float)  # None (not reached) -> NaN\n    steps.to_csv(out / 'cfcheck_steps.csv', index=False)\n    cells.to_csv(out / 'cfcheck_cells.csv', index=False)\n    summary = {'config': config, 'soil_baseline': dict(zip(BANDS, soil['values'])),\n               **summarize_check(cells, steps, cell),\n               'forward_passes': passes, 'seconds': seconds, 'seconds_per_cell': seconds / len(cells)}\n    (out / 'cfcheck_summary.json').write_text(json.dumps(clean(summary), indent=1))\n    print(json.dumps(clean({k: summary[k] for k in ('cells', 'rescored_decision_differs', 'k_not_spray',\n                                                    'k_no_spray', 'reverts_to_spray', 'curve')}), indent=1))\n\n\ndef main():\n    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)\n    ap.add_argument('--data', required=True)\n    ap.add_argument('--ckpt', required=True)\n    ap.add_argument('--decisions', required=True, help='decisions.csv written by src.weeds.explain for this model')\n    ap.add_argument('--soil', required=True, help='soil_baseline.json written by src.weeds.explain')\n    ap.add_argument('--out', required=True)\n    ap.add_argument('--cell', type=int, default=PRIMARY['cell_size'])\n    ap.add_argument('--patch', type=int, default=PATCH)\n    ap.add_argument('--extra', type=int, default=2, help='patches scanned past NO_SPRAY and both estimates')\n    ap.add_argument('--batch', type=int, default=8)\n    ap.add_argument('--no-amp', dest='amp', action='store_false', help='float32 throughout (slower, more memory)')\n    ap.add_argument('--budget-hours', type=float, default=3.0)\n    ap.add_argument('--limit-tiles', type=int, default=0, help='check only the first N tiles (smoke test)')\n    args = ap.parse_args()\n    dev = torch.device('cuda' if torch.cuda.is_available() else 'cpu')\n    print('device:', dev, torch.cuda.get_device_name(0) if dev.type == 'cuda' else '')\n    run(args, dev)\n\n\nif __name__ == '__main__':\n    main()\n"}
for rel, text in FILES.items():
    path = WORK / rel
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text, encoding='utf-8')
os.chdir(WORK)
sys.path.insert(0, str(WORK))
print('Đã ghi', len(FILES), 'file mã nguồn vào', WORK)
print('Thư mục kết quả:', RUN)

import subprocess
def run(*args):
    """Run a module of the pilot, stream its output, and stop 'Run all' if it fails."""
    p = subprocess.Popen([sys.executable, '-u', '-m', *map(str, args)], stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end='')
    if p.wait() != 0:
        raise RuntimeError(f'Lệnh lỗi (mã {p.returncode}). Dừng lại và gửi thông báo lỗi ở trên cho Claude.')

## 2. Phần A: tải và chuẩn bị dữ liệu (bỏ qua nếu đã có từ các notebook trước)

In [ ]:
import subprocess
from src.weeds.data import build_npz

ZIP = DATA_DRIVE / 'weedsgalore-dataset.zip'
if not ZIP.exists() or ZIP.stat().st_size != 336989804:
    subprocess.run(['curl', '-L', '--retry', '3', '-o', str(ZIP), 'https://doidata.gfz.de/weedsgalore_e_celikkan_2024/weedsgalore-dataset.zip'], check=True)
assert ZIP.stat().st_size == 336989804, f'Kích thước file zip không đúng: {ZIP.stat().st_size}'

if all((DATA_DRIVE / f'weedsgalore_{s}.npz').exists() for s in ('train', 'val', 'test')):
    print('Dữ liệu gọn đã có trên Drive, bỏ qua.')
else:
    build_npz(ZIP, DATA_DRIVE)

## 3. Phần B: kiểm tra GPU và chép dữ liệu về máy Colab

Trước mục này: *Runtime → Change runtime type → T4 GPU*, rồi chạy lại mục 1.

In [ ]:
import shutil, torch
!nvidia-smi -L
print('PyTorch', torch.__version__, '| CUDA:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Chưa có GPU: đổi runtime sang T4 rồi chạy lại mục 1.'
DATA = Path('/content/data')
DATA.mkdir(exist_ok=True)
for s in ('train', 'val', 'test'):
    src = DATA_DRIVE / f'weedsgalore_{s}.npz'
    assert src.exists(), f'Chưa có {src}: chạy Phần A (mục 2) trước.'
    if not (DATA / src.name).exists():
        shutil.copy(src, DATA)
print(sorted(p.name for p in DATA.iterdir()))

In [ ]:
import time
from src.weeds.runs import RUNS, by_schedule, summarize, train_args

# the session clock counts from the start of this Colab VM, so running the notebook again on it does not reset it
T0 = time.time() - float(Path('/proc/uptime').read_text().split()[0])
SESSION_HOURS = 3.5  # training stops in time to test, analyse, package and sync to Drive before the Colab limit
PILOT = BASE / 'run_dlv3p_msi_seed0_b8'  # run 2
print(f'Máy Colab này đã chạy {(time.time() - T0) / 3600:.2f} giờ; huấn luyện dừng khi đủ {SESSION_HOURS} giờ.')

## 4. Chạy thử (chỉ ở phiên đầu, vài phút)

Chạy 1 epoch chỉ 2 vòng lặp với learning rate giảm dần, rồi đánh giá cả mô hình tốt nhất lẫn mô hình epoch cuối, để bắt lỗi trước khi huấn luyện thật. Từ phiên thứ hai trở đi, mục này tự bỏ qua.

In [ ]:
if not any((RUN / r['name'] / 'history.json').exists() for r in RUNS):
    shutil.rmtree('/content/smoke', ignore_errors=True)
    run('src.weeds.train', '--data', DATA, '--out', '/content/smoke', '--smoke', '--batch', 8, '--lr-schedule', 'cosine')
    run('src.weeds.train', '--stage', 'test', '--weights', 'last', '--smoke', '--data', DATA, '--out', '/content/smoke')
else:
    print('Đã có lần chạy bắt đầu từ phiên trước: bỏ qua chạy thử.')

## 5. Huấn luyện, đánh giá và tầng quyết định cho từng lần chạy

Mỗi epoch in một dòng như notebook thí nghiệm thử. Lần đã xong được bỏ qua. Khi một lần huấn luyện xong: đánh giá mô hình tốt nhất (`best.pt`, giữ lại) và mô hình epoch cuối (kết quả trong thư mục con `last/`), rồi xóa checkpoint `last.pt` (khoảng 0,5 GB).

In [ ]:
def evaluate_models(out):
    """Test and decision layer for the model chosen on validation and for the final-epoch model (each step once)."""
    if not (out / 'go_no_go.json').exists():
        run('src.weeds.train', '--stage', 'test', '--data', DATA, '--out', out)
        run('src.weeds.analyze', '--data', DATA, '--out', out)
    if (out / 'last.pt').exists() and not (out / 'last' / 'go_no_go.json').exists():
        run('src.weeds.train', '--stage', 'test', '--weights', 'last', '--data', DATA, '--out', out)
        run('src.weeds.analyze', '--data', DATA, '--out', out / 'last')

# run 2: only its final-epoch model is new (its checkpoint is kept: it belongs to the earlier notebook)
if (PILOT / 'last.pt').exists() and not (PILOT / 'last' / 'go_no_go.json').exists():
    print('========== lần chạy 2: mô hình epoch cuối ==========')
    run('src.weeds.train', '--stage', 'test', '--weights', 'last', '--data', DATA, '--out', PILOT)
    run('src.weeds.analyze', '--data', DATA, '--out', PILOT / 'last')

for r in RUNS:
    out = RUN / r['name']
    if not (out / 'train_done.json').exists():
        left = SESSION_HOURS - (time.time() - T0) / 3600
        if left < 0.3:
            print(f'Hết thời gian của phiên này (còn {left:.2f} giờ). Phiên sau: Run all lại, các lần đã xong tự bỏ qua.')
            break
        print()
        print(f"========== {r['name']} | phiên còn {left:.2f} giờ ==========")
        run('src.weeds.train', '--stage', 'train', '--data', DATA, '--out', out, *train_args(r),
            '--budget-hours', f'{left - 0.15:.2f}')  # 0.15 h kept for the two evaluations and packaging
        if not (out / 'train_done.json').exists():
            print(f"{r['name']}: chưa huấn luyện xong, sẽ chạy tiếp từ checkpoint ở phiên sau.")
            break
    evaluate_models(out)  # also completes a run whose evaluation an earlier session left unfinished
    for f in ('last.pt', 'last.tmp'):
        (out / f).unlink(missing_ok=True)
    print(f"{r['name']}: xong.")

## 6. Tổng hợp các lần chạy đã xong (kèm lần chạy 2)

In [ ]:
import pandas as pd
pd.set_option('display.width', 220)
dirs = {'es_seed0 (lần chạy 2)': PILOT, **{r['name']: RUN / r['name'] for r in RUNS}}
df = summarize(dirs)
if df.empty:
    print('Chưa có lần chạy nào xong.')
else:
    print(df.round(3).to_string(index=False))
    print()
    print('mIoU test theo lịch huấn luyện và cách chọn mô hình (bài báo và mô hình công bố của tác giả: 82,90):')
    print(by_schedule(df).round(2).to_string(index=False))
    df.to_csv(RUN / 'runs_summary.csv', index=False)
if df.attrs['skipped']:
    print('Chưa có kết quả:', ', '.join(df.attrs['skipped']))

## 7. Đóng gói kết quả (chỉ file json và csv; mô hình và xác suất dự đoán ở lại trên Drive)

In [ ]:
import zipfile
ZIP_OUT = BASE / 'ket_qua_weeds_runs.zip'
with zipfile.ZipFile(ZIP_OUT, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in sorted(RUN.rglob('*')):
        if f.suffix in ('.json', '.csv'):
            z.write(f, RESULTS / f.relative_to(RUN))
    for f in sorted((PILOT / 'last').glob('*')):  # run 2's final-epoch model
        if f.suffix in ('.json', '.csv'):
            z.write(f, RESULTS / 'pilot_b8_seed0' / 'last' / f.name)
print('Đã tạo', ZIP_OUT, f'({ZIP_OUT.stat().st_size / 1e6:.1f} MB)')

## 8. Ghi hết dữ liệu lên Drive (chờ dòng "Đã ghi hết" rồi mới đóng phiên)

In [ ]:
drive.flush_and_unmount()  # checkpoints of 0.5 GB may still be syncing; the next session resumes from them
print('Đã ghi hết dữ liệu lên Drive. Có thể đóng phiên Colab.')

**Bước tiếp theo:** khi cả 6 lần đã xong (hoặc muốn xem kết quả giữa chừng), tải `ket_qua_weeds_runs.zip` về, giải nén vào thư mục gốc của repo (sẽ tạo thư mục `results/weeds_runs/`), rồi nhờ Claude phân tích.